### VAE training for real data

V1: 2026/05/31     
V3: 2026/06/02 added balanced cluster modeling     
V4: 2026/06/04 updated preprocess function

In [ ]:
#!pip install scikit-optimize

In [2]:
import json
import os
from pathlib import Path
from math import sqrt
import xarray as xr
import pandas as pd
import numpy as np
from collections import defaultdict
from umap import UMAP
import pickle


import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt 

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn.functional as F

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MeanShift, estimate_bandwidth
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
from scipy.signal import savgol_filter, find_peaks
from scipy.stats import mode
import hdbscan
import seaborn as sns

import cv2

import gc
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args


#### Prepare training data

In [3]:


# ── just point to your folder ─────────────────────────────────────────────────
json_folder = Path(r"E:\ferg_take2\json_files")   # Path() + raw string (r"...")
json_files  = list(json_folder.glob("*.json"))
print(f"Found {len(json_files)} json files")

all_arrays = []
body_parts = None

for json_file in json_files:
    with open(json_file) as f:
        data = json.load(f)

    annotations = data["annotations"]
    frame_ids   = list(annotations.keys())
    n_frames    = len(frame_ids)
    first_key   = frame_ids[0]

    if body_parts is None:
        body_parts = list(annotations[first_key].keys())
    else:
        assert list(annotations[first_key].keys()) == body_parts, \
            f"Body parts mismatch in {json_file}"

    n_bodyparts = len(body_parts)
    arr = np.zeros((n_frames, n_bodyparts, 2), dtype=np.float32)

    for fi, fid in enumerate(frame_ids):
        for bi, bp in enumerate(body_parts):
            arr[fi, bi, :] = annotations[fid][bp]

    all_arrays.append(arr)
    print(f"Loaded {json_file.name} — {n_frames} frames")

# ── concatenate everything into one array ─────────────────────────────────────
arr_all = np.concatenate(all_arrays, axis=0)

da = xr.DataArray(
    arr_all,
    dims=["frame", "bodypart", "coord"],
    coords={
        "frame":    np.arange(len(arr_all)),
        "bodypart": body_parts,
        "coord":    ["x", "y"],
    },
    name="keypoints",
)

print("\nDataArray shape:", da.shape)

nc_file = "./data/exp_data.nc"
da.to_netcdf(nc_file)


Found 28 json files
Loaded F10 Baseline_predictions.json — 17165 frames
Loaded F10DOITest_predictions.json — 15697 frames
Loaded F10Withdrawal_predictions.json — 17808 frames
Loaded F2 Baseline_predictions.json — 16981 frames
Loaded F2DOITest_predictions.json — 17354 frames
Loaded F2withdrawal._predictions.json — 15106 frames
Loaded F3 Baseline_predictions.json — 16418 frames
Loaded F3DOITest_predictions.json — 15899 frames
Loaded F3Withdrawal_predictions.json — 16080 frames
Loaded F4 Baseline_predictions.json — 16349 frames
Loaded F4DOITest_predictions.json — 15080 frames
Loaded F4Withdrawal_predictions.json — 16600 frames
Loaded F5 Baseline_predictions.json — 16731 frames
Loaded F5DOITest_predictions.json — 17279 frames
Loaded F5Withdrawal_predictions.json — 17329 frames
Loaded F6 Baseline_predictions.json — 16513 frames
Loaded F6DOITest_predictions.json — 17235 frames
Loaded F6WithdrawalP2_predictions.json — 6834 frames
Loaded F6WithdrawalPart1_predictions.json — 11196 frames
Loaded

Load prepared dataset from local disk

In [4]:
nc_file = "./data/exp_data.nc"
da = xr.open_dataarray(nc_file)
print("\nDataArray shape:", da.shape)


DataArray shape: (447182, 7, 2)


In [5]:
# V5 — adds absolute arena position as auxiliary features
# alongside the existing egocentric representation (does NOT
# replace egocentric alignment — just adds position on top).

def preprocess_with_position(raw_sequence):
    # compute speed from raw arena movement BEFORE centering
    center_raw  = raw_sequence[:, 1, :]                        # (T, 2)
    center_diff = np.diff(center_raw, axis=0)                  # (T-1, 2)
    center_diff = np.concatenate([center_diff[:1], center_diff], axis=0)  # (T, 2)
    speed       = np.linalg.norm(center_diff, axis=-1)         # (T,)
    print(f"Speed stats: mean={speed.mean():.3f}, std={speed.std():.3f}, max={speed.max():.3f}")
    speed       = np.tile(speed[:, None, None], (1, 7, 1))     # (T, 7, 1)

    # center and heading-align
    centroid = raw_sequence[:, 1:2, :]
    centered = raw_sequence - centroid

    head  = centered[:, 0, :]
    angle = np.arctan2(head[:, 1], head[:, 0])
    cos_a = np.cos(-angle)
    sin_a = np.sin(-angle)

    x = centered[:, :, 0]
    y = centered[:, :, 1]
    x_rot = x * cos_a[:, None] - y * sin_a[:, None]
    y_rot = x * sin_a[:, None] + y * cos_a[:, None]
    aligned = np.stack([x_rot, y_rot], axis=-1)                # (T, 7, 2)

    # velocity of aligned joints (captures gait/paw swing)
    vel = np.diff(aligned, axis=0)
    vel = np.concatenate([vel[:1], vel], axis=0)               # (T, 7, 2)

    # angular velocity (captures turning)
    ang_vel = np.diff(angle, axis=0)
    ang_vel = np.concatenate([ang_vel[:1], ang_vel])
    ang_vel = np.tile(ang_vel[:, None, None], (1, 7, 1))       # (T, 7, 1)

    # ── NEW: absolute arena position (centroid), tiled across joints ──
    # This is deliberately NOT rotated/aligned — it's the raw arena-frame
    # (x, y) of the animal's centroid, so the model can learn proximity
    # to fixed arena landmarks (lever, spout) if that's discriminative.
    abs_position = np.tile(center_raw[:, None, :], (1, 7, 1))  # (T, 7, 2)

    print(f"Absolute position stats — x: mean={center_raw[:,0].mean():.2f}, "
          f"std={center_raw[:,0].std():.2f}; "
          f"y: mean={center_raw[:,1].mean():.2f}, std={center_raw[:,1].std():.2f}")

    return np.concatenate(
        [aligned, vel, ang_vel, speed, abs_position], axis=-1
    )  # (T, 7, 8) — was (T, 7, 6), now +2 for (x, y) position

raw_processed = preprocess_with_position(da.values)
print(f"New raw_processed shape: {raw_processed.shape}")  # should be (T, 7, 8)


Speed stats: mean=5.627, std=38.768, max=1593.495
Absolute position stats — x: mean=929.06, std=411.26; y: mean=407.87, std=158.11
New raw_processed shape: (447182, 7, 8)


In [6]:
n_frames, n_joints, n_coords = raw_processed.shape
da_scaled = np.zeros_like(raw_processed)
scalers = []

for j in range(n_joints):
    joint_scalers = []
    for c in range(n_coords):
        scaler = StandardScaler()
        da_scaled[:, j, c] = scaler.fit_transform(
            raw_processed[:, j, c].reshape(-1, 1)
        ).squeeze()
        joint_scalers.append(scaler)
    scalers.append(joint_scalers)

raw_processed = da_scaled

In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

#### Classes and functions for modeling

In [8]:
# ============================================================
# MODEL
# ============================================================

class HierarchicalRAE(nn.Module):
    def __init__(self,
                 joint_dim,
                 joint_embed=32,
                 pose_embed=128,
                 hidden_dim=256,
                 latent_dim=64,
                 num_joints=7):
        super().__init__()
        self.num_joints = num_joints

        # --- Encoder ---
        self.joint_encoder = nn.Sequential(
            nn.Linear(joint_dim, joint_embed),
            nn.Tanh(),
            nn.Linear(joint_embed, joint_embed)
        )
        self.pose_encoder = nn.Sequential(
            nn.Linear(num_joints * joint_embed, pose_embed),
            nn.Tanh()
        )
        self.encoder_rnn = nn.LSTM(pose_embed, hidden_dim, batch_first=True)
        self.fc_latent    = nn.Linear(hidden_dim, latent_dim)

        # --- Decoder ---
        self.fc_decode_h  = nn.Linear(latent_dim, hidden_dim)
        self.fc_decode_c  = nn.Linear(latent_dim, hidden_dim)
        self.decoder_rnn  = nn.LSTM(latent_dim, hidden_dim, batch_first=True)
        self.decoder_proj = nn.Linear(hidden_dim, pose_embed)
        self.pose_decoder = nn.Sequential(
            nn.Linear(pose_embed, num_joints * joint_embed),
            nn.Tanh()
        )
        self.joint_decoder = nn.Linear(joint_embed, joint_dim)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc_decode_h.weight, gain=2.0)
        nn.init.xavier_uniform_(self.fc_decode_c.weight, gain=2.0)
        nn.init.constant_(self.fc_decode_h.bias, 0.0)
        nn.init.constant_(self.fc_decode_c.bias, 0.0)
        nn.init.xavier_uniform_(self.fc_latent.weight, gain=2.0)
        nn.init.constant_(self.fc_latent.bias, 0.0)

        for name, param in self.decoder_rnn.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param, gain=2.0)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param, gain=2.0)
            elif 'bias' in name:
                nn.init.zeros_(param)

        for name, param in self.encoder_rnn.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)

        for module in [self.joint_encoder, self.pose_encoder,
                       self.pose_decoder, self.joint_decoder,
                       self.decoder_proj]:
            if isinstance(module, nn.Sequential):
                for layer in module:
                    if isinstance(layer, nn.Linear):
                        nn.init.xavier_uniform_(layer.weight)
                        nn.init.zeros_(layer.bias)
            elif isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def encode(self, x, lengths=None):
        from torch.nn.utils.rnn import pack_padded_sequence
        B, T, J, C = x.shape

        x_flat = x.view(B * T * J, C)
        x_flat = self.joint_encoder(x_flat)
        x_enc  = x_flat.view(B, T, J, -1)
        x_enc  = x_enc.view(B, T, -1)
        x_enc  = self.pose_encoder(x_enc)

        if lengths is not None:
            packed = pack_padded_sequence(
                x_enc, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            _, (h, _) = self.encoder_rnn(packed)
        else:
            _, (h, _) = self.encoder_rnn(x_enc)

        z = self.fc_latent(h[-1])
        return z

    def decode(self, z, T):
        B   = z.shape[0]
        h   = self.fc_decode_h(z).unsqueeze(0)
        c   = self.fc_decode_c(z).unsqueeze(0)
        inp = z.unsqueeze(1).repeat(1, T, 1)
        dec, _ = self.decoder_rnn(inp, (h, c))
        dec    = self.decoder_proj(dec)
        return dec

    def forward(self, x, lengths=None):
        B, T, J, C = x.shape
        z   = self.encode(x, lengths)
        dec = self.decode(z, T)
        dec = self.pose_decoder(dec)
        dec = dec.view(B, T, J, -1)
        dec = dec.view(B * T * J, -1)
        dec = self.joint_decoder(dec)
        dec = dec.view(B, T, J, C)
        return dec, z


# ============================================================
# COLLATE FUNCTION
# ============================================================

def collate_variable_length(batch):
    lengths = [x.shape[0] for x in batch]
    B       = len(batch)
    T_max   = max(lengths)
    J, C    = batch[0].shape[1], batch[0].shape[2]

    padded = torch.zeros(B, T_max, J, C)
    for i, x in enumerate(batch):
        padded[i, :lengths[i]] = x

    return padded, torch.tensor(lengths, dtype=torch.long)


# ============================================================
# EXTRACT WINDOWS (fixed-size, Stage 1 only)
# ============================================================

def extract_nonoverlapping_windows(raw_sequence, window_size):
    n_frames = len(raw_sequence)
    windows  = []
    for start in range(0, n_frames - window_size, window_size):
        windows.append(raw_sequence[start:start + window_size])
    windows = np.array(windows)
    print(f"Extracted {len(windows)} non-overlapping windows")
    print(f"Coverage: {len(windows) * window_size}/{n_frames} frames "
          f"({100 * len(windows) * window_size / n_frames:.1f}%)")
    return windows



#### Training starts here

In [9]:
MAX_ITER = 15
EPOCHS = 2000
VW_EPOCHS = 1000
BATCH_SIZE = 256     #640   # 1280    # 1536  # 128X12
LR = 1e-3
WINDOW_SIZE = 30

PERCENTILE_RANGE = (20,75)   #(45.0, 70.0)
QUANTILE_RANGE = (0.05,0.20)   # (0.05, 0.2)
LR_RANGE = (1e-4, 1e-2)

Function definition

In [10]:

# ============================================================
# STAGE 1: Train on fixed-size windows
# ============================================================

def train_on_fixed_windows(raw_sequence, window_size=WINDOW_SIZE,
                            epochs=EPOCHS, batch_size=BATCH_SIZE,
                            lr=LR, device=device,
                            patience=30):
    windows  = extract_nonoverlapping_windows(raw_sequence, window_size)
    X_tensor = torch.tensor(windows, dtype=torch.float32)

    n_joints  = raw_sequence.shape[1]
    joint_dim = raw_sequence.shape[2]

    from torch.utils.data import TensorDataset
    dataset = TensorDataset(X_tensor)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model     = HierarchicalRAE(latent_dim=30, joint_dim=joint_dim,
                                num_joints=n_joints).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min', factor=0.5, patience=10, verbose=False)
    loss_fn   = nn.MSELoss()

    best_loss    = float('inf')
    patience_ctr = 0
    best_weights = None
    lossgraph    = []

    print("Training on fixed windows...")
    for epoch in range(epochs):
        total_loss = 0
        for (batch,) in loader:
            batch    = batch.to(device)
            recon, _ = model(batch)
            loss     = loss_fn(recon, batch)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        avg = total_loss / len(loader)
        lossgraph.append(avg)
        scheduler.step(avg)

        if avg < best_loss - 1e-4:
            best_loss    = avg
            patience_ctr = 0
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1

        if patience_ctr >= patience:
            print(f"Early stopping at epoch {epoch+1} — best loss: {best_loss:.6f}")
            model.load_state_dict(best_weights)
            break

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} — Loss: {avg:.6f}  "
                  f"patience: {patience_ctr}/{patience}")

    return model, lossgraph


# ============================================================
# STAGE 4b: Retrain on variable-length windows
# ============================================================

def train_on_variable_windows(raw_sequence, windows,
                               epochs=VW_EPOCHS, batch_size=BATCH_SIZE,
                               lr=LR, device=device,
                               patience=30):
    
    
    
    n_joints  = raw_sequence.shape[1]
    joint_dim = raw_sequence.shape[2]

    tensor_list = [torch.tensor(w, dtype=torch.float32) for w in windows]
    loader = DataLoader(tensor_list, batch_size=batch_size, shuffle=True,
                        collate_fn=collate_variable_length)

    model     = HierarchicalRAE(latent_dim=30, joint_dim=joint_dim,
                                num_joints=n_joints).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min', factor=0.5, patience=10, verbose=False)
    loss_fn   = nn.MSELoss()

    best_loss    = float('inf')
    patience_ctr = 0
    best_weights = None
    lossgraph    = []

    print("Retraining on variable-length windows...")
    for epoch in range(epochs):
        total_loss = 0
        for padded, lengths in loader:
            padded   = padded.to(device)
            recon, _ = model(padded, lengths=lengths)
            loss = torch.tensor(0.0, device=device)
            for i, l in enumerate(lengths):
                loss = loss + loss_fn(recon[i, :l], padded[i, :l])
            loss = loss / len(lengths)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        avg = total_loss / len(loader)
        lossgraph.append(avg)
        scheduler.step(avg)

        if avg < best_loss - 1e-4:
            best_loss    = avg
            patience_ctr = 0
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1

        if patience_ctr >= patience:
            print(f"Early stopping at epoch {epoch+1} — best loss: {best_loss:.6f}")
            model.load_state_dict(best_weights)
            break

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} — Loss: {avg:.6f}  "
                  f"patience: {patience_ctr}/{patience}")

    return model, lossgraph

# ============================================================
# STAGE 2: Compute reconstruction loss signal
# ============================================================

def compute_frame_loss(model, raw_sequence, window_size,
                        stride=5, device=device):
    model.eval()
    losses    = []
    positions = []

    raw_tensor = torch.tensor(raw_sequence, dtype=torch.float32, device=device)
    loss_fn    = nn.MSELoss()

    with torch.no_grad():
        for start in range(0, len(raw_sequence) - window_size, stride):
            window   = raw_tensor[start:start + window_size].unsqueeze(0)
            recon, _ = model(window)
            loss     = loss_fn(recon, window).item()
            losses.append(loss)
            positions.append(start + window_size // 2)

            if (start) % 100000 == 0:
                print(start,end="|")

    positions = np.array(positions)
    losses    = np.array(losses)
    print(f"Computed loss at {len(losses)} positions")
    print(f"Loss stats — mean: {losses.mean():.4f}, "
          f"std: {losses.std():.4f}, max: {losses.max():.4f}")
    return positions, losses


# ============================================================
# STAGE 3: Detect transitions
# ============================================================

def find_transitions(positions, losses,
                      percentile=70, smoothing=5,
                      min_distance=3, fps=30):
    if len(losses) < smoothing:
        smoothing = max(3, len(losses) // 2)
        if smoothing % 2 == 0:
            smoothing += 1

    smoothed  = savgol_filter(losses, window_length=smoothing, polyorder=2)
    threshold = np.percentile(smoothed, percentile)
    peaks, _  = find_peaks(smoothed, height=threshold, distance=min_distance)
    transition_frames = positions[peaks]

    if len(transition_frames) > 1:
        intervals = np.diff(transition_frames)
        mean_bout = np.mean(intervals) / fps
        print(f"Found {len(transition_frames)} transitions")
        print(f"Mean bout duration: {mean_bout:.2f}s")
        if mean_bout < 1:
            print("WARNING: bouts too short — raise percentile or min_distance")
        elif mean_bout > 15:
            print("WARNING: bouts too long — lower percentile")
        else:
            print("Bout duration looks plausible")

    return transition_frames, smoothed


# ============================================================
# STAGE 4: Variable-length windows from segments
# ============================================================

def create_windows_from_transitions(raw_sequence, transition_frames,
                                     min_segment_frames=15,
                                     max_segment_frames=300):
    n_frames   = len(raw_sequence)
    boundaries = np.unique(
        np.concatenate([[0], transition_frames, [n_frames]])
    ).astype(int)

    all_windows   = []
    window_labels = []

    for seg_idx in range(len(boundaries) - 1):
        seg_start = boundaries[seg_idx]
        seg_end   = boundaries[seg_idx + 1]
        seg_len   = seg_end - seg_start

        if seg_len < min_segment_frames:
            continue

        segment = raw_sequence[seg_start:seg_end]

        for start in range(0, seg_len, max_segment_frames):
            chunk = segment[start:start + max_segment_frames]
            if len(chunk) < min_segment_frames:
                continue
            all_windows.append(chunk)
            window_labels.append(seg_idx)

    lengths = [len(w) for w in all_windows]
    print(f"Created {len(all_windows)} variable-length windows from "
          f"{len(boundaries) - 1} segments")
    print(f"Window lengths — min: {min(lengths)}, "
          f"max: {max(lengths)}, mean: {np.mean(lengths):.1f}")

    return all_windows, np.array(window_labels)


# ============================================================
# STAGE 5: Encode, UMAP, cluster
# ============================================================

def encode_and_cluster(model, windows, batch_size=BATCH_SIZE,
                        device=device, quantile=0.1,
                        umap_neighbors=30, umap_min_dist=0.1):
    model.eval()
    all_latents = []

    tensor_list = [torch.tensor(w, dtype=torch.float32) for w in windows]
    loader = DataLoader(tensor_list, batch_size=batch_size, shuffle=False,
                        collate_fn=collate_variable_length)

    with torch.no_grad():
        for padded, lengths in loader:
            padded = padded.to(device)
            _, z   = model(padded, lengths=lengths)
            all_latents.append(z.cpu().numpy())

    all_latents = np.concatenate(all_latents, axis=0)  # (N, 16)
    print(f"Latents shape: {all_latents.shape}")

    # UMAP: 16D → 2D
    print("Running UMAP...")
    reducer    = UMAP(n_components=2, n_neighbors=umap_neighbors,
                      min_dist=umap_min_dist, metric='cosine',
                      random_state=42)
    latents_2d = reducer.fit_transform(all_latents)     # (N, 2)
    print(f"UMAP done. Shape: {latents_2d.shape}")

    # MeanShift on 2D UMAP embedding
    normed    = normalize(latents_2d, norm='l2')
    bandwidth = estimate_bandwidth(normed, quantile=quantile)
    print(f"Estimated bandwidth: {bandwidth:.4f}")

    ms     = MeanShift(bandwidth=bandwidth, bin_seeding=True)
    ms.fit(normed)
    labels = ms.labels_

    n_clusters = len(np.unique(labels))
    print(f"Found {n_clusters} clusters")
    print(f"Cluster sizes: {np.bincount(labels)}")

    if n_clusters > 1:
        dist_matrix = np.clip(1 - np.dot(normed, normed.T), 0, 2)
        sil = silhouette_score(dist_matrix, labels, metric='precomputed')
        print(f"Silhouette score: {sil:.4f}")

    return all_latents, latents_2d, labels, ms


# ============================================================
# FULL TWO-PASS PIPELINE
# ============================================================

def run_pipeline_test(raw_sequence, window_size, fps,
                      epochs, percentile, quantile,
                      min_segment_frames, max_segment_frames,
                      stride=5, umap_neighbors=30,
                      umap_min_dist=0.1, device=device):

    # ── PASS 1: fixed windows → transition detection ───────────────────
    print("\n" + "="*50)
    print("STAGE 1: Training RAE on fixed windows")
    print("="*50)
    n_joints  = raw_sequence.shape[1]
    joint_dim = raw_sequence.shape[2]

    model_stage1 = HierarchicalRAE(latent_dim=30, joint_dim=joint_dim,
                                    num_joints=n_joints).to(device)
    state_dict = torch.load('model_stage1.pth', weights_only=True)
    model_stage1.load_state_dict(state_dict)
    print("model_stage_1 loaded...")

    print("\n" + "="*50)
    print("STAGE 2: Computing reconstruction loss signal")
    print("="*50)
    positions = np.load('fix_win_positions.npy')
    losses = np.load('fix_win_losses.npy')
    print("position and loss loaded...")

    print("\n" + "="*50)
    print("STAGE 3: Finding behavioral transitions")
    print("="*50)
    transition_frames, smoothed = find_transitions(
        positions, losses,
        percentile=percentile, fps=fps
    )

    print("\n" + "="*50)
    print("STAGE 4: Creating variable-length behavioral windows")
    print("="*50)
    windows, window_segment_labels = create_windows_from_transitions(
        raw_sequence, transition_frames,
        min_segment_frames=min_segment_frames,
        max_segment_frames=max_segment_frames
    )

    # ── PASS 2: retrain on variable-length windows ─────────────────────
    print("\n" + "="*50)
    print("STAGE 4b: Retraining RAE on variable-length windows")
    print("="*50)
    model_stage2, lossgraph_stage2 = train_on_variable_windows(
        raw_sequence, windows,
        epochs=epochs, lr=LR, device=device
    )

    print("\n" + "="*50)
    print("STAGE 5: Encoding + UMAP + clustering")
    print("="*50)
    latents, latents_2d, cluster_labels, ms_model = encode_and_cluster(
        model_stage2, windows,
        quantile=quantile,
        umap_neighbors=umap_neighbors,
        umap_min_dist=umap_min_dist,
        device=device
    )

    return {
        'model':                 model_stage2,
        'model_stage1':          model_stage1,
        'latents':               latents,        # (N, 16) — raw high-dim latents
        'latents_2d':            latents_2d,     # (N, 2)  — UMAP projection
        'cluster_labels':        cluster_labels,
        'transition_frames':     transition_frames,
        'windows':               windows,
        'window_segment_labels': window_segment_labels,
        'losses':                losses,
        'positions':             positions,
        'lossgraph_stage1':      lossgraph_stage1,
        'lossgraph_stage2':      lossgraph_stage2,
        'smoothed_losses':       smoothed,
    }

In [11]:
# ==============================================================================
# HYPERPARAMETER TUNING: Bayesian optimization over percentile, quantile, (& lr)
# ==============================================================================

# ── Train Stage 1 ONCE (shared across all iterations) ────────
print("="*60)
print("PRE-STEP: Training Stage 1 model (shared across iterations)")
print("="*60)
model_stage1, lossgraph_stage1 = train_on_fixed_windows(
    raw_processed, window_size=WINDOW_SIZE, epochs=EPOCHS
)




PRE-STEP: Training Stage 1 model (shared across iterations)
Extracted 14906 non-overlapping windows
Coverage: 447180/447182 frames (100.0%)


c:\Users\fbai_\anaconda3\envs\condaenv312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training on fixed windows...
Epoch 5/2000 — Loss: 0.493270  patience: 0/30
Epoch 10/2000 — Loss: 0.471188  patience: 0/30
Epoch 15/2000 — Loss: 0.462836  patience: 1/30
Epoch 20/2000 — Loss: 0.442912  patience: 0/30
Epoch 25/2000 — Loss: 0.432934  patience: 0/30
Epoch 30/2000 — Loss: 0.419013  patience: 2/30
Epoch 35/2000 — Loss: 0.397178  patience: 0/30
Epoch 40/2000 — Loss: 0.385567  patience: 0/30
Epoch 45/2000 — Loss: 0.371899  patience: 2/30
Epoch 50/2000 — Loss: 0.366772  patience: 1/30
Epoch 55/2000 — Loss: 0.359053  patience: 3/30
Epoch 60/2000 — Loss: 0.338484  patience: 0/30
Epoch 65/2000 — Loss: 0.328532  patience: 0/30
Epoch 70/2000 — Loss: 0.335333  patience: 2/30
Epoch 75/2000 — Loss: 0.328966  patience: 4/30
Epoch 80/2000 — Loss: 0.322866  patience: 1/30
Epoch 85/2000 — Loss: 0.324103  patience: 6/30
Epoch 90/2000 — Loss: 0.312346  patience: 4/30
Epoch 95/2000 — Loss: 0.308959  patience: 1/30
Epoch 100/2000 — Loss: 0.301001  patience: 0/30
Epoch 105/2000 — Loss: 0.306901

In [12]:
print("\n" + "="*60)
print("PRE-STEP: Computing reconstruction loss signal (shared)")
print("="*60)
positions, losses = compute_frame_loss(
    model_stage1, raw_processed, window_size=WINDOW_SIZE, stride=5, device=device
)

np.save('fix_win_positions.npy', positions)
np.save('fix_win_losses.npy', losses)


PRE-STEP: Computing reconstruction loss signal (shared)
0|100000|200000|300000|400000|Computed loss at 89431 positions
Loss stats — mean: 0.3425, std: 1.5883, max: 68.4657


In [13]:
with open("lossgraph_stage1.json", "w") as file:
    json.dump(lossgraph_stage1, file)

Load posistions, losses and model_stage_1 from local disk

In [14]:
positions = np.load('fix_win_positions.npy')
losses = np.load('fix_win_losses.npy')

In [15]:
n_joints  = raw_processed.shape[1]
joint_dim = raw_processed.shape[2]

model_stage1 = HierarchicalRAE(latent_dim=64, joint_dim=joint_dim,
                                num_joints=n_joints).to(device)


Bayesian search for optimal percentile and quantile 

In [16]:
MAX_ITER = 15
EPOCHS = 2000
VW_EPOCHS = 1500
BATCH_SIZE = 256     #640   # 1280    # 1536  # 128X12
LR = 1e-3
WINDOW_SIZE = 30

PERCENTILE_RANGE = (20,75)   #(45.0, 70.0)
QUANTILE_RANGE = (0.05,0.20)   # (0.05, 0.2)
LR_RANGE = (1e-4, 1e-2)

In [17]:
# ── Define search space ──────────────────────────────────────
search_space = [
    Real(*PERCENTILE_RANGE, name='percentile'),
    Real(*QUANTILE_RANGE,   name='quantile'),
    #Real(*LR_RANGE,         name='lr', prior='log-uniform'),
]

search_log = []
best_results = None
best_score = -1
iteration = 0

In [18]:
# ── Objective function ───────────────────────────────────────
@use_named_args(search_space)
def objective(percentile, quantile, lr=LR):
    global iteration, best_score, best_results

    iteration += 1
    print(f"\n{'='*60}")
    print(f"ITERATION {iteration}/{MAX_ITER}  |  percentile={percentile:.2f}, quantile={quantile:.4f}, lr={lr:.6f}")
    print(f"{'='*60}")

    # Stage 3: transitions
    transition_frames, smoothed = find_transitions(
        positions, losses, percentile=percentile, fps=30
    )

    # Stage 4: variable windows
    windows, window_segment_labels = create_windows_from_transitions(
        raw_processed, transition_frames,
        min_segment_frames=60, max_segment_frames=600
    )

    # Stage 4b: retrain on variable windows (with tuned lr)
    model_file_name = "model_stage2_"+str(iteration)+".pth"
    if Path(model_file_name).is_file():
        n_joints  = raw_processed.shape[1]
        joint_dim = raw_processed.shape[2]

        model_stage2 = HierarchicalRAE(latent_dim=30, joint_dim=joint_dim,
                                num_joints=n_joints).to(device)
        state_dict = torch.load(model_file_name, weights_only=True)
        model_stage2.load_state_dict(state_dict)
        print("model_satge2 loaded...")
    else:
        model_stage2, lossgraph_stage2 = train_on_variable_windows(
            raw_processed, windows, epochs=VW_EPOCHS, lr=lr, device=device
        )
        torch.save(model_stage2.state_dict(), os.path.join('./', model_file_name))

    # Stage 5: encode + cluster
    latents, latents_2d, cluster_labels, ms_model = encode_and_cluster(
        model_stage2, windows, quantile=quantile,
        umap_neighbors=30, umap_min_dist=0.1, device=device
    )

    n_clusters = len(np.unique(cluster_labels))

    # ── Compute silhouette score ──────────────────────────────
    # silhouette needs at least 2 clusters and more samples than clusters
    if n_clusters < 2 or n_clusters >= len(latents_2d):
        print(f"  → {n_clusters} clusters — skipping silhouette (invalid cluster count)")
        score = -1.0
    else:
        score = silhouette_score(latents_2d, cluster_labels)

    log_entry = {
        "iteration": iteration,
        "percentile": round(percentile, 2),
        "quantile": round(quantile, 4),
        "lr": round(float(lr), 6),
        "n_clusters": n_clusters,
        "silhouette": round(float(score), 4),
    }
    search_log.append(log_entry)
    print(f"  → {n_clusters} clusters, silhouette={score:.4f}")

    # Track best
    if score > best_score:
        best_score = score
        best_results = {
            'model': model_stage2,
            'model_stage1': model_stage1,
            'latents': latents,
            'latents_2d': latents_2d,
            'cluster_labels': cluster_labels,
            'transition_frames': transition_frames,
            'windows': windows,
            'window_segment_labels': window_segment_labels,
            'losses': losses,
            'positions': positions,
            'lossgraph_stage1': lossgraph_stage1,
            #'lossgraph_stage2': lossgraph_stage2,
            'smoothed_losses': smoothed,
            '_percentile': percentile,
            '_quantile': quantile,
            '_lr': lr,
        }
        print(f"  ** NEW BEST (silhouette={score:.4f}) **")

    # Cleanup GPU memory
    del model_stage2, latents, latents_2d, cluster_labels, ms_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # gp_minimize minimizes, so return negative score
    return -score

In [19]:
# ── Run Bayesian optimization ────────────────────────────────
bayes_result = gp_minimize(
    func=objective,
    dimensions=search_space,
    n_calls=MAX_ITER,
    n_initial_points=min(MAX_ITER, 5),
    random_state=42,
    verbose=False,
)


ITERATION 1/15  |  percentile=63.81, quantile=0.0775, lr=0.001000
Found 6180 transitions
Mean bout duration: 2.41s
Bout duration looks plausible
Created 2146 variable-length windows from 6181 segments
Window lengths — min: 60, max: 600, mean: 155.0
Retraining on variable-length windows...


c:\Users\fbai_\anaconda3\envs\condaenv312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 10/1500 — Loss: 0.359759  patience: 0/30
Epoch 20/1500 — Loss: 0.338387  patience: 0/30
Epoch 30/1500 — Loss: 0.336143  patience: 2/30
Epoch 40/1500 — Loss: 0.320721  patience: 0/30
Epoch 50/1500 — Loss: 0.314187  patience: 0/30
Epoch 60/1500 — Loss: 0.307518  patience: 0/30
Epoch 70/1500 — Loss: 0.313693  patience: 2/30
Epoch 80/1500 — Loss: 0.298813  patience: 2/30
Epoch 90/1500 — Loss: 0.293314  patience: 1/30
Epoch 100/1500 — Loss: 0.288514  patience: 0/30
Epoch 110/1500 — Loss: 0.291258  patience: 1/30
Epoch 120/1500 — Loss: 0.283287  patience: 0/30
Epoch 130/1500 — Loss: 0.279587  patience: 3/30
Epoch 140/1500 — Loss: 0.281038  patience: 3/30
Epoch 150/1500 — Loss: 0.278674  patience: 8/30
Epoch 160/1500 — Loss: 0.272469  patience: 0/30
Epoch 170/1500 — Loss: 0.269727  patience: 7/30
Epoch 180/1500 — Loss: 0.264524  patience: 1/30
Epoch 190/1500 — Loss: 0.269492  patience: 3/30
Epoch 200/1500 — Loss: 0.263381  patience: 8/30
Epoch 210/1500 — Loss: 0.258810  patience: 0/30
E

c:\Users\fbai_\anaconda3\envs\condaenv312\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP done. Shape: (2146, 2)
Estimated bandwidth: 0.1107
Found 12 clusters
Cluster sizes: [546 339 422 265 190 160 101 108   8   4   2   1]
Silhouette score: 0.7554
  → 12 clusters, silhouette=0.2748
  ** NEW BEST (silhouette=0.2748) **

ITERATION 2/15  |  percentile=62.88, quantile=0.1395, lr=0.001000
Found 6304 transitions
Mean bout duration: 2.36s
Bout duration looks plausible
Created 2156 variable-length windows from 6305 segments
Window lengths — min: 60, max: 600, mean: 152.7
Retraining on variable-length windows...


...

UMAP done. Shape: (2251, 2)
Estimated bandwidth: 0.3130
Found 7 clusters
Cluster sizes: [1380  487  128   87   71   68   30]
Silhouette score: 0.8724
  → 7 clusters, silhouette=0.5223

ITERATION 15/15  |  percentile=22.29, quantile=0.1695, lr=0.001000
Found 10186 transitions
Mean bout duration: 1.46s
Bout duration looks plausible
Created 2165 variable-length windows from 10187 segments
Window lengths — min: 60, max: 585, mean: 95.8
Retraining on variable-length windows...


c:\Users\fbai_\anaconda3\envs\condaenv312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 10/1500 — Loss: 0.248258  patience: 0/30
Epoch 20/1500 — Loss: 0.232983  patience: 1/30
Epoch 30/1500 — Loss: 0.226052  patience: 0/30
Epoch 40/1500 — Loss: 0.221780  patience: 3/30
Epoch 50/1500 — Loss: 0.219556  patience: 1/30
Epoch 60/1500 — Loss: 0.218787  patience: 4/30
Epoch 70/1500 — Loss: 0.214143  patience: 4/30
Epoch 80/1500 — Loss: 0.209957  patience: 0/30
Epoch 90/1500 — Loss: 0.207540  patience: 3/30
Epoch 100/1500 — Loss: 0.204006  patience: 0/30
Epoch 110/1500 — Loss: 0.203088  patience: 1/30
Epoch 120/1500 — Loss: 0.199241  patience: 0/30
Epoch 130/1500 — Loss: 0.201920  patience: 1/30
Epoch 140/1500 — Loss: 0.197498  patience: 4/30
Epoch 150/1500 — Loss: 0.193570  patience: 0/30
Epoch 160/1500 — Loss: 0.198793  patience: 10/30
Epoch 170/1500 — Loss: 0.190287  patience: 4/30
Epoch 180/1500 — Loss: 0.188295  patience: 14/30
Epoch 190/1500 — Loss: 0.185560  patience: 0/30
Epoch 200/1500 — Loss: 0.187374  patience: 9/30
Epoch 210/1500 — Loss: 0.184302  patience: 7/30

c:\Users\fbai_\anaconda3\envs\condaenv312\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP done. Shape: (2165, 2)
Estimated bandwidth: 0.1065
Found 4 clusters
Cluster sizes: [662 614 465 424]
Silhouette score: 0.8455
  → 4 clusters, silhouette=0.4161


In [20]:
# ── Summary ──────────────────────────────────────────────────
print(f"\n{'='*60}")
print("HYPERPARAMETER SEARCH COMPLETE")
print(f"{'='*60}")
print(f"\nSearch log:")
for entry in search_log:
    marker = " <-- BEST" if entry['silhouette'] == best_score else ""
    print(f"  Iter {entry['iteration']:2d}: percentile={entry['percentile']:5.2f}, "
          f"quantile={entry['quantile']:.4f}, lr={entry['lr']:.6f}  "
          f"→  {entry['n_clusters']} clusters, silhouette={entry['silhouette']:.4f}{marker}")

print(f"\nBest result:")
print(f"  percentile : {best_results['_percentile']:.2f}")
print(f"  quantile   : {best_results['_quantile']:.4f}")
print(f"  lr         : {best_results['_lr']:.6f}")
print(f"  clusters   : {len(np.unique(best_results['cluster_labels']))}")
print(f"  silhouette : {best_score:.4f}")

# Set results to best for downstream cells
results = best_results

# Save the model
torch.save(results['model'].state_dict(), os.path.join('./', "model_stage2.pth"))



HYPERPARAMETER SEARCH COMPLETE

Search log:
  Iter  1: percentile=63.81, quantile=0.0775, lr=0.001000  →  12 clusters, silhouette=0.2748
  Iter  2: percentile=62.88, quantile=0.1395, lr=0.001000  →  5 clusters, silhouette=0.4828
  Iter  3: percentile=44.52, quantile=0.0650, lr=0.001000  →  11 clusters, silhouette=0.1619
  Iter  4: percentile=45.26, quantile=0.1001, lr=0.001000  →  8 clusters, silhouette=0.3266
  Iter  5: percentile=27.86, quantile=0.1476, lr=0.001000  →  5 clusters, silhouette=0.4441
  Iter  6: percentile=75.00, quantile=0.1476, lr=0.001000  →  7 clusters, silhouette=0.3661
  Iter  7: percentile=20.72, quantile=0.1980, lr=0.001000  →  4 clusters, silhouette=0.4899
  Iter  8: percentile=72.42, quantile=0.2000, lr=0.001000  →  4 clusters, silhouette=0.4829
  Iter  9: percentile=20.00, quantile=0.2000, lr=0.001000  →  4 clusters, silhouette=0.4821
  Iter 10: percentile=47.03, quantile=0.1890, lr=0.001000  →  4 clusters, silhouette=0.5591
  Iter 11: percentile=51.74, quan

In [ ]:
import matplotlib.colors as mcolors
embedded = results['latents']
labels = results['cluster_labels']
plt.figure(figsize=(8, 6))
plt.scatter(embedded[:, 0], embedded[:, 1])
plt.show()


"""latents = results['latents']
normed  = normalize(latents, norm='l2')
labels = results['cluster_labels']
# UMAP to 2D
reducer   = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.5, random_state=42)
embedded  = reducer.fit_transform(normed)
"""
embedded = results['latents']
labels = results['cluster_labels']

unique_labels = np.sort(np.unique(labels))
n_clusters = len(unique_labels)

cmap = plt.get_cmap('tab10', n_clusters)
norm = mcolors.BoundaryNorm(np.arange(n_clusters + 1) - 0.5, n_clusters)

plt.figure(figsize=(8, 6))
sc = plt.scatter(embedded[:, 0], embedded[:, 1],
                  c=labels, cmap=cmap, norm=norm, s=15, alpha=0.8)

cbar = plt.colorbar(sc, label='Cluster', ticks=unique_labels)
cbar.ax.set_yticklabels([str(int(l)) for l in unique_labels])

plt.show()

In [ ]:
import pickle

save_path = r'E:\ferg_take2\carve_results.pkl'  # <-- hardcode your path here

with open(save_path, 'wb') as f:
    pickle.dump(results, f)

print(f"Saved to {save_path}")

In [ ]:
import pickle
import shutil

# CARVE's output — latents, cluster_labels, windows, and whatever else 'active' holds
with open('/home/claude/carve_results.pkl', 'wb') as f:
    pickle.dump(active, f)

# copy to outputs so it's downloadable
shutil.copy('/home/claude/carve_results.pkl', '/mnt/user-data/outputs/carve_results.pkl')

print(f"Saved. Keys: {list(active.keys())}")

In [ ]:
import re

CONDITION_PATTERNS = {
    'baseline':   re.compile(r'baseline', re.IGNORECASE),
    'DOI':        re.compile(r'doi', re.IGNORECASE),
    'withdrawal': re.compile(r'withdrawal', re.IGNORECASE),
}

def get_condition(json_name):
    for cond, pat in CONDITION_PATTERNS.items():
        if pat.search(json_name):
            return cond
    return 'unknown'

def get_rat_id(json_name):
    # adjust this regex to however rat IDs are encoded, e.g. "F2" in F2DOITest
    m = re.match(r'([A-Za-z]+\d+)', json_name)
    return m.group(1) if m else json_name

In [ ]:
results=new_best_results

In [ ]:
import matplotlib.colors as mcolors
import numpy as np

# NOTE: results is a list of 14 dicts, so you need to index into it first.
# Replace results_idx with the correct index (or swap this whole block for `active`
# if you want the same clustering we've been using throughout this conversation).
results_idx = 0  # <-- set this to the entry you want

# ── build rat identity per window, same way we did earlier in this conversation ──
rat_ids = np.array([
    get_rat_id(frame_to_video[starts[i]][0]) for i in range(len(embedded))
])

unique_rats = np.sort(np.unique(rat_ids))
n_rats = len(unique_rats)

cmap = plt.get_cmap('tab10', n_rats)
rat_to_idx = {rat: i for i, rat in enumerate(unique_rats)}
color_idx = np.array([rat_to_idx[r] for r in rat_ids])

norm = mcolors.BoundaryNorm(np.arange(n_rats + 1) - 0.5, n_rats)

plt.figure(figsize=(8, 6))
sc = plt.scatter(embedded[:, 0], embedded[:, 1],
                  c=color_idx, cmap=cmap, norm=norm, s=15, alpha=0.8)

cbar = plt.colorbar(sc, label='Rat', ticks=np.arange(n_rats))
cbar.ax.set_yticklabels(unique_rats)

plt.title('Latent space colored by rat identity')
plt.show()

In [27]:
import matplotlib.colors as mcolors
import numpy as np

# NOTE: results is a list of 14 dicts, so you need to index into it first.
# Replace results_idx with the correct entry (or swap this block for `active`
# if you want the same clustering we've been using throughout this conversation).
results_idx = 0  # <-- set this to the entry you want


# ── build condition per window, same way we did earlier in this conversation ──
conditions = np.array([
    get_condition(frame_to_video[starts[i]][0]) for i in range(len(embedded))
])

cond_order = ['baseline', 'DOI', 'withdrawal']
cond_colors = {'baseline': 'tab:blue', 'DOI': 'tab:red', 'withdrawal': 'tab:green'}

plt.figure(figsize=(8, 6))
for cond in cond_order:
    mask = conditions == cond
    plt.scatter(embedded[mask, 0], embedded[mask, 1],
                color=cond_colors[cond], label=cond, s=15, alpha=0.6)

plt.legend()
plt.xlabel('Latent dim 1')
plt.ylabel('Latent dim 2')
plt.title('Latent space colored by condition')
plt.show()

NameError: name 'embedded' is not defined

In [ ]:
# ── group composition analysis + bar chart ──────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
# --- group definitions, based on real filenames like:
#   M11 Baseline_predictions.json
#   M11DOITest_predictions.json
#   M14DOITestPart1_predictions.json / M14DOITestPart2_predictions.json
#   F4 Withdrawal_predictions.json
GROUP_KEYWORDS = {
    "DOI":          ["DOITest"],
    "withdrawal":   ["Withdrawal"],
    "baseline":     ["Baseline"],
}
def get_group(json_name):
    """Return the group label for a window's source json file, or None if no match."""
    name = json_name.lower()
    for group in ["DOI", "withdrawal", "baseline"]:
        keywords = GROUP_KEYWORDS[group]
        if any(kw.lower() in name for kw in keywords):
            return group
    return None

# --- build a group label for every window — no windows excluded ---
window_groups = []
for win_idx in range(len(results['windows'])):
    abs_start = starts[win_idx]
    json_name, _ = frame_to_video[abs_start]  # use window's starting frame's source file
    group = get_group(json_name)
    window_groups.append(group)
window_groups = np.array(window_groups, dtype=object)

n_unmatched = np.sum(window_groups == None)
print(f"Windows with no matched group (other): {n_unmatched} / {len(window_groups)}")

# --- compute per-cluster group composition (all windows included) ---
labels      = results['cluster_labels']
n_clusters  = len(np.unique(results['cluster_labels']))  # keep original cluster numbering
group_names = list(GROUP_KEYWORDS.keys())
rows = []
for c in range(n_clusters):
    cluster_mask   = (labels == c)
    cluster_groups = window_groups[cluster_mask]
    n_total        = len(cluster_groups)
    row = {"cluster": c, "n_windows": n_total}
    for g in group_names:
        n_g  = np.sum(cluster_groups == g)
        pct  = 100 * n_g / n_total if n_total > 0 else 0.0
        row[f"{g}_n"]   = n_g
        row[f"{g}_pct"] = round(pct, 1)
    row["unmatched_n"] = np.sum(cluster_groups == None)
    rows.append(row)
composition_df = pd.DataFrame(rows)
print(composition_df.to_string(index=False))
composition_df.to_csv(output_folder / "cluster_group_composition.csv", index=False)
print(f"\nSaved → {output_folder / 'cluster_group_composition.csv'}")

# --- stacked bar chart ---
group_order  = ["baseline", "DOI", "withdrawal"]
group_colors = {
    "baseline":    "#8c9c8c",  # muted sage green
    "DOI":         "#b98b8b",  # muted rose
    "withdrawal":  "#8c9db9",  # muted blue
}
clusters = composition_df["cluster"].values
x        = np.arange(len(clusters))
fig, ax = plt.subplots(figsize=(max(8, len(clusters) * 0.6), 5))
bottom = np.zeros(len(clusters))
for g in group_order:
    pct_col = f"{g}_pct"
    vals    = composition_df[pct_col].values
    ax.bar(x, vals, bottom=bottom, label=g.replace("_", " "),
           color=group_colors[g], edgecolor="white", linewidth=0.5)
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels([f"Cluster {c}" for c in clusters], rotation=45, ha="right")
ax.set_ylabel("% of windows")
ax.set_ylim(0, 100)
ax.set_title("Group Composition by Cluster")
ax.legend(title="Group", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
fig_path = output_folder / "cluster_group_composition.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved → {fig_path}")

In [ ]:
import re
import numpy as np
import pandas as pd
from itertools import groupby

# ── helper functions ────────────────────────────────────────────────────
CONDITION_PATTERNS = {
    'baseline':   re.compile(r'baseline', re.IGNORECASE),
    'DOI':        re.compile(r'doi', re.IGNORECASE),
    'withdrawal': re.compile(r'withdrawal', re.IGNORECASE),
}

def get_condition(json_name):
    for cond, pat in CONDITION_PATTERNS.items():
        if pat.search(json_name):
            return cond
    return 'unknown'

def get_rat_id(json_name):
    m = re.match(r'([A-Za-z]+\d+)', json_name)
    return m.group(1) if m else json_name

# ── rebuild df_full: per-window cluster/session table, in temporal order ──
window_json_full    = []
window_cluster_full = []
window_start_full   = []

for win_idx, cluster_label in enumerate(results['cluster_labels']):
    json_name, _ = frame_to_video[starts[win_idx]]
    window_json_full.append(json_name)
    window_cluster_full.append(cluster_label)
    window_start_full.append(starts[win_idx])

df_full = pd.DataFrame({
    'json_name': window_json_full,
    'cluster':   window_cluster_full,
    'start':     window_start_full,
})

# ── rebuild df: same but with condition/rat, unknown-condition rows dropped ──
df = df_full.copy()
df['condition'] = df['json_name'].apply(get_condition)
df['rat'] = df['json_name'].apply(get_rat_id)
df = df[df['condition'] != 'unknown']

# ── rebuild bout/transition features per session ──────────────────────────
cluster_ids = sorted(df_full['cluster'].unique())
n_clusters = len(cluster_ids)

def session_bout_features(cluster_seq, cluster_ids):
    seq = list(cluster_seq)
    n_windows = len(seq)
    bouts = [(k, len(list(g))) for k, g in groupby(seq)]

    bout_durations = {c: [] for c in cluster_ids}
    for c, length in bouts:
        bout_durations[c].append(length)

    mean_bout_dur = {}
    n_bouts = {}
    for c in cluster_ids:
        durs = bout_durations[c]
        mean_bout_dur[c] = np.mean(durs) if durs else 0.0
        n_bouts[c] = len(durs)

    n_transitions = sum(1 for i in range(1, n_windows) if seq[i] != seq[i-1])
    transition_rate = n_transitions / max(n_windows - 1, 1)

    trans_counts = np.zeros((n_clusters, n_clusters))
    idx_map = {c: i for i, c in enumerate(cluster_ids)}
    for i in range(1, n_windows):
        a, b = idx_map[seq[i-1]], idx_map[seq[i]]
        trans_counts[a, b] += 1
    row_sums = trans_counts.sum(axis=1, keepdims=True)
    trans_probs = np.divide(trans_counts, row_sums, out=np.zeros_like(trans_counts), where=row_sums != 0)

    with np.errstate(divide='ignore', invalid='ignore'):
        row_entropy = -np.nansum(np.where(trans_probs > 0, trans_probs * np.log2(trans_probs), 0), axis=1)
    mean_transition_entropy = np.nanmean(row_entropy) if row_sums.sum() > 0 else 0.0

    feats = {}
    for c in cluster_ids:
        feats[f'mean_bout_dur_{c}'] = mean_bout_dur[c]
        feats[f'n_bouts_{c}'] = n_bouts[c]
    feats['transition_rate'] = transition_rate
    feats['mean_transition_entropy'] = mean_transition_entropy
    return feats

rows = []
for json_name, sess in df_full.groupby('json_name'):
    sess_sorted = sess.sort_values('start')
    seq = sess_sorted['cluster'].values
    feats = session_bout_features(seq, cluster_ids)
    feats['json_name'] = json_name
    rows.append(feats)

bout_feat_df = pd.DataFrame(rows).set_index('json_name')

# ── occupancy fractions ────────────────────────────────────────────────
occupancy_df = (
    df_full.groupby('json_name')['cluster']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reindex(columns=cluster_ids, fill_value=0)
)
occupancy_df.columns = [f'occ_{c}' for c in occupancy_df.columns]

combined_features = occupancy_df.join(bout_feat_df, how='inner')

# ── attach condition + rat metadata, rebuild combined and X ───────────────
session_meta = df.groupby('json_name').agg(condition=('condition', 'first'), rat=('rat', 'first'))
combined = combined_features.join(session_meta, how='inner')

feature_cols = combined_features.columns.tolist()
X = combined[feature_cols].values
y = combined['condition'].values
groups = combined['rat'].values

print(f"Rebuilt X: shape {X.shape}, y: {len(y)} labels, groups: {len(np.unique(groups))} rats")

In [ ]:
print(len(feature_cols))
print(feature_cols)

In [ ]:
import numpy as np
import pandas as pd
from itertools import product

# ── subset combined to DOI/withdrawal only, using the current (17-feature) combined table ──
mask_2class = combined['condition'].isin(['DOI', 'withdrawal'])
combined_2class = combined[mask_2class].copy()

# average multiple sessions per rat+condition (handles F6's duplicate), same as before
combined_2class_avg = (
    combined_2class
    .reset_index()
    .groupby(['rat', 'condition'])[feature_cols]
    .mean()
    .reset_index()
)

# keep only rats with both conditions present (drops F1, which has no withdrawal session)
rat_counts = combined_2class_avg['rat'].value_counts()
complete_rats = rat_counts[rat_counts == 2].index
print(f"Rats with both conditions: {len(complete_rats)} (dropped: {set(rat_counts.index) - set(complete_rats)})")

combined_2class_complete = combined_2class_avg[combined_2class_avg['rat'].isin(complete_rats)].copy()
combined_2class_complete = combined_2class_complete.sort_values(['rat', 'condition']).reset_index(drop=True)

# ── pivot into paired DOI/withdrawal columns per rat ───────────────────────
pivot_17 = combined_2class_complete.pivot(index='rat', columns='condition', values=feature_cols)
pivot_17 = pivot_17.dropna()
print(f"Paired rats: {len(pivot_17)}")

# ── build the paired difference matrix: one row per rat, one column per feature ──
diff_matrix = np.array([
    pivot_17[(feat, 'DOI')].values - pivot_17[(feat, 'withdrawal')].values
    for feat in feature_cols
]).T  # shape: (n_rats, n_features)

n_rats, n_features = diff_matrix.shape
print(f"Paired rats: {n_rats}, features: {n_features}")

# ── test statistic: norm of the standardized mean difference vector ──────
def test_statistic(diffs):
    feat_std = diffs.std(axis=0, ddof=1)
    feat_std[feat_std == 0] = 1  # avoid divide-by-zero for constant features
    standardized_mean_diff = diffs.mean(axis=0) / feat_std
    return np.linalg.norm(standardized_mean_diff)

observed_stat = test_statistic(diff_matrix)
print(f"Observed test statistic: {observed_stat:.4f}")

# ── exact permutation: enumerate all 2^n_rats sign-flip combinations ──────
if n_rats <= 20:
    null_stats = []
    for signs in product([1, -1], repeat=n_rats):
        signs = np.array(signs).reshape(-1, 1)
        permuted_diffs = diff_matrix * signs
        null_stats.append(test_statistic(permuted_diffs))
    null_stats = np.array(null_stats)

    p_value = (np.sum(null_stats >= observed_stat)) / len(null_stats)
    print(f"\nExact permutation test ({len(null_stats)} sign-flip combinations, 17-feature set):")
    print(f"  Observed statistic = {observed_stat:.4f}")
    print(f"  Null mean = {null_stats.mean():.4f} ± {null_stats.std():.4f}")
    print(f"  Exact p-value = {p_value:.4f}")
else:
    print("Too many rats for exact enumeration — use Monte Carlo sign-flip sampling instead.")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
logo = LeaveOneGroupOut()

C_values = [0.001, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
mean_accs = []

for C in C_values:
    clf = LogisticRegression(solver='saga', C=C, l1_ratio=1, max_iter=5000)
    scores = cross_val_score(clf, X_scaled, y, groups=groups, cv=logo, scoring='accuracy')
    mean_accs.append(scores.mean())
    print(f"C={C:<6}  LOGO accuracy = {scores.mean():.3f}")

best_C = C_values[int(np.argmax(mean_accs))]
print(f"\nBest C by mean LOGO accuracy: {best_C}")

clf = LogisticRegression(solver='saga', C=best_C, l1_ratio=1, max_iter=5000)
y_pred = cross_val_predict(clf, X_scaled, y, groups=groups, cv=logo)

print(f"\nLeave-one-rat-out accuracy (L1, C={best_C}): {accuracy_score(y, y_pred):.3f}")
print(classification_report(y, y_pred))


cmap = LinearSegmentedColormap.from_list(
    'custom', ["#f3f1bc", "#350526"]
)
cond_order = ['baseline', 'DOI', 'withdrawal']
cm = confusion_matrix(y, y_pred, labels=cond_order)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
            xticklabels=cond_order, yticklabels=cond_order)
plt.xlabel('Predicted condition')
plt.ylabel('True condition')
plt.title(f'L1-regularized (C={best_C})\n(leave-one-rat-out CV)')
plt.tight_layout()
plt.show()

clf_full = LogisticRegression(solver='saga', C=best_C, l1_ratio=1, max_iter=5000)
clf_full.fit(X_scaled, y)

coef_df = pd.DataFrame(clf_full.coef_, index=clf_full.classes_, columns=feature_cols)
nonzero_mask = (coef_df.abs() > 1e-6).any(axis=0)
surviving_features = coef_df.columns[nonzero_mask].tolist()
print(f"\n{len(surviving_features)} / {len(feature_cols)} features survived L1:")
print(surviving_features)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# ── reuse X, y, groups, feature_cols, best_C from the L1 pipeline ─────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
logo = LeaveOneGroupOut()

n_permutations = 1000
rng = np.random.default_rng(seed=0)

def run_logo_l1(X_scaled, y_labels, groups, C):
    clf = LogisticRegression(solver='saga', C=C, l1_ratio=1, max_iter=5000)
    y_pred = cross_val_predict(clf, X_scaled, y_labels, groups=groups, cv=logo)
    acc = accuracy_score(y_labels, y_pred)
    f1_macro = f1_score(y_labels, y_pred, average='macro')
    return acc, f1_macro

# ── true (observed) performance ────────────────────────────────────────
true_acc, true_f1 = run_logo_l1(X_scaled, y, groups, best_C)
print(f"Observed LOGO accuracy: {true_acc:.3f}")
print(f"Observed LOGO macro-F1: {true_f1:.3f}")

# ── null distribution: shuffle condition labels, rerun full pipeline ──────
# shuffling y directly (not within-rat) tests the null that cluster features
# carry no information about condition at all, which is the relevant null here
null_accs = []
null_f1s  = []

for i in range(n_permutations):
    y_shuffled = rng.permutation(y)
    acc, f1_macro = run_logo_l1(X_scaled, y_shuffled, groups, best_C)
    null_accs.append(acc)
    null_f1s.append(f1_macro)
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{n_permutations} permutations done...")

null_accs = np.array(null_accs)
null_f1s  = np.array(null_f1s)

# ── p-values: fraction of null runs >= observed ────────────────────────
p_acc = (np.sum(null_accs >= true_acc) + 1) / (n_permutations + 1)
p_f1  = (np.sum(null_f1s  >= true_f1)  + 1) / (n_permutations + 1)

print(f"\nPermutation test results ({n_permutations} permutations):")
print(f"  Observed accuracy = {true_acc:.3f}, null mean = {null_accs.mean():.3f} ± {null_accs.std():.3f}, p = {p_acc:.4f}")
print(f"  Observed macro-F1 = {true_f1:.3f}, null mean = {null_f1s.mean():.3f} ± {null_f1s.std():.3f}, p = {p_f1:.4f}")

# ── plot null distribution vs observed ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(null_accs, bins=30, color='gray', alpha=0.7, label='Null (shuffled labels)')
axes[0].axvline(true_acc, color='red', linewidth=2, label=f'Observed = {true_acc:.3f}')
axes[0].axvline(1/3, color='black', linestyle='--', linewidth=1, label='Chance (1/3)')
axes[0].set_xlabel('LOGO accuracy')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Accuracy permutation test\np = {p_acc:.4f}')
axes[0].legend(fontsize=9)

axes[1].hist(null_f1s, bins=30, color='gray', alpha=0.7, label='Null (shuffled labels)')
axes[1].axvline(true_f1, color='red', linewidth=2, label=f'Observed = {true_f1:.3f}')
axes[1].set_xlabel('LOGO macro-F1')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Macro-F1 permutation test\np = {p_f1:.4f}')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(output_folder / 'permutation_test.png', dpi=150)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# ── subset to DOI and withdrawal sessions only ─────────────────────────────
mask_2class = combined['condition'].isin(['DOI', 'withdrawal'])
combined_2class = combined[mask_2class]

X_2class = combined_2class[feature_cols].values
y_2class = combined_2class['condition'].values
groups_2class = combined_2class['rat'].values

print(f"Sessions: {len(y_2class)} ({pd.Series(y_2class).value_counts().to_dict()})")
print(f"Rats: {len(np.unique(groups_2class))}")

scaler_2class = StandardScaler()
X_2class_scaled = scaler_2class.fit_transform(X_2class)

logo_2class = LeaveOneGroupOut()

# ── C sweep for this subset ────────────────────────────────────────────────
C_values = [0.001, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
mean_accs_2class = []

for C in C_values:
    clf = LogisticRegression(solver='saga', C=C, l1_ratio=1, max_iter=5000)
    scores = cross_val_score(clf, X_2class_scaled, y_2class, groups=groups_2class,
                              cv=logo_2class, scoring='accuracy')
    mean_accs_2class.append(scores.mean())
    print(f"C={C:<6}  LOGO accuracy = {scores.mean():.3f}")

best_C_2class = C_values[int(np.argmax(mean_accs_2class))]
print(f"\nBest C: {best_C_2class}")

# ── final fit + predictions ────────────────────────────────────────────────
clf_2class = LogisticRegression(solver='saga', C=best_C_2class, l1_ratio=1, max_iter=5000)
y_pred_2class = cross_val_predict(clf_2class, X_2class_scaled, y_2class,
                                   groups=groups_2class, cv=logo_2class)

acc_2class = accuracy_score(y_2class, y_pred_2class)
print(f"\nLeave-one-rat-out accuracy (DOI vs withdrawal, C={best_C_2class}): {acc_2class:.3f}")
print(classification_report(y_2class, y_pred_2class))

cond_order_2class = ['DOI', 'withdrawal']
cm_2class = confusion_matrix(y_2class, y_pred_2class, labels=cond_order_2class)

plt.figure(figsize=(5, 4.5))
sns.heatmap(cm_2class, annot=True, fmt='d', cmap=cmap,
            xticklabels=cond_order_2class, yticklabels=cond_order_2class)
plt.xlabel('Predicted condition')
plt.ylabel('True condition')
plt.title(f'DOI vs withdrawal (C={best_C_2class})\n(leave-one-rat-out CV)')
plt.tight_layout()
plt.savefig(output_folder / 'doi_vs_withdrawal_confusion.png', dpi=150)
plt.show()

# ── surviving features ──────────────────────────────────────────────────
clf_2class_full = LogisticRegression(solver='saga', C=best_C_2class, l1_ratio=1, max_iter=5000)
clf_2class_full.fit(X_2class_scaled, y_2class)

coef_2class_df = pd.DataFrame(
    clf_2class_full.coef_, columns=feature_cols
).T
coef_2class_df.columns = ['coefficient']
nonzero_2class = coef_2class_df[coef_2class_df['coefficient'].abs() > 1e-6]
print(f"\n{len(nonzero_2class)} / {len(feature_cols)} features survived L1:")
print(nonzero_2class.sort_values('coefficient'))

# ── permutation test ────────────────────────────────────────────────────
n_permutations = 1000
rng = np.random.default_rng(seed=0)

def run_logo_l1_2class(X_s, y_labels, groups, C):
    clf = LogisticRegression(solver='saga', C=C, l1_ratio=1, max_iter=5000)
    y_pred = cross_val_predict(clf, X_s, y_labels, groups=groups, cv=logo_2class)
    acc = accuracy_score(y_labels, y_pred)
    f1_binary = f1_score(y_labels, y_pred, pos_label='DOI')
    return acc, f1_binary

true_acc_2class, true_f1_2class = run_logo_l1_2class(
    X_2class_scaled, y_2class, groups_2class, best_C_2class
)
print(f"\nObserved accuracy: {true_acc_2class:.3f}, F1 (DOI): {true_f1_2class:.3f}")

null_accs_2class = []
null_f1s_2class = []
for i in range(n_permutations):
    y_shuffled = rng.permutation(y_2class)
    acc, f1_val = run_logo_l1_2class(X_2class_scaled, y_shuffled, groups_2class, best_C_2class)
    null_accs_2class.append(acc)
    null_f1s_2class.append(f1_val)
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{n_permutations} permutations done...")

null_accs_2class = np.array(null_accs_2class)
null_f1s_2class = np.array(null_f1s_2class)

p_acc_2class = (np.sum(null_accs_2class >= true_acc_2class) + 1) / (n_permutations + 1)
p_f1_2class = (np.sum(null_f1s_2class >= true_f1_2class) + 1) / (n_permutations + 1)

print(f"\nPermutation test (DOI vs withdrawal, {n_permutations} permutations):")
print(f"  Observed accuracy = {true_acc_2class:.3f}, null mean = {null_accs_2class.mean():.3f} ± {null_accs_2class.std():.3f}, p = {p_acc_2class:.4f}")
print(f"  Observed F1 = {true_f1_2class:.3f}, null mean = {null_f1s_2class.mean():.3f} ± {null_f1s_2class.std():.3f}, p = {p_f1_2class:.4f}")

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

# ── build a paired table: one row per rat, columns for DOI and withdrawal ──
occ_cols = [c for c in feature_cols if c.startswith('occ_')]
other_cols = [c for c in feature_cols if not c.startswith('occ_')]

# average multiple sessions per rat+condition (handles F6's duplicate DOI/withdrawal session)
combined_2class_avg = (
    combined_2class
    .reset_index()
    .groupby(['rat', 'condition'])[feature_cols]
    .mean()
    .reset_index()
)

pivot = combined_2class_avg.pivot(index='rat', columns='condition', values=feature_cols)

# drop any rat missing either condition (removes F1, which has no withdrawal session)
pivot = pivot.dropna()
print(f"Paired rats after dropping incomplete pairs: {len(pivot)}")
print(f"Rats included: {list(pivot.index)}")

# ── paired test for each feature ────────────────────────────────────────
results = []
for feat in feature_cols:
    doi_vals = pivot[(feat, 'DOI')].values
    wd_vals  = pivot[(feat, 'withdrawal')].values
    diffs = doi_vals - wd_vals

    # paired t-test
    t_stat, t_p = stats.ttest_rel(doi_vals, wd_vals)
    # paired non-parametric alternative (more robust with small n)
    try:
        w_stat, w_p = stats.wilcoxon(doi_vals, wd_vals)
    except ValueError:
        w_stat, w_p = np.nan, np.nan

    results.append({
        'feature': feat,
        'mean_DOI': doi_vals.mean(),
        'mean_withdrawal': wd_vals.mean(),
        'mean_diff (DOI - withdrawal)': diffs.mean(),
        'paired_t_p': t_p,
        'wilcoxon_p': w_p,
    })

results_df = pd.DataFrame(results).sort_values('paired_t_p')
pd.set_option('display.width', 120)
print("\nPaired comparison, all features (sorted by paired t-test p-value):")
print(results_df.to_string(index=False))

# ── multiple comparisons correction (14 tests) ──────────────────────────
_, pvals_corrected, _, _ = multipletests(results_df['paired_t_p'], method='fdr_bh')
results_df['paired_t_p_fdr'] = pvals_corrected
print("\nWith FDR-corrected p-values:")
print(results_df[['feature', 'mean_diff (DOI - withdrawal)', 'paired_t_p', 'paired_t_p_fdr']].to_string(index=False))

# ── plot: paired dot plot for each occupancy feature (cluster-level) ──────
fig, axes = plt.subplots(1, len(occ_cols), figsize=(4 * len(occ_cols), 5), sharey=True)
if len(occ_cols) == 1:
    axes = [axes]

for ax, feat in zip(axes, occ_cols):
    doi_vals = pivot[(feat, 'DOI')].values
    wd_vals  = pivot[(feat, 'withdrawal')].values
    for d, w in zip(doi_vals, wd_vals):
        ax.plot([0, 1], [d, w], color='gray', alpha=0.5, linewidth=1)
    ax.scatter(np.zeros_like(doi_vals), doi_vals, color='crimson', zorder=3, label='DOI')
    ax.scatter(np.ones_like(wd_vals), wd_vals, color='steelblue', zorder=3, label='withdrawal')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['DOI', 'withdrawal'])
    p_val = results_df.loc[results_df['feature'] == feat, 'paired_t_p'].values[0]
    ax.set_title(f"{feat}\np={p_val:.3f}")
    ax.set_xlim(-0.3, 1.3)

axes[0].set_ylabel('Occupancy fraction')
axes[0].legend()
plt.tight_layout()
plt.savefig(output_folder / 'doi_vs_withdrawal_occupancy_paired.png', dpi=150)
plt.show()

# ── same paired plot for the bout/transition features ─────────────────────
n_other = len(other_cols)
n_cols_plot = 4
n_rows_plot = int(np.ceil(n_other / n_cols_plot))
fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(4 * n_cols_plot, 4 * n_rows_plot), sharey=False)
axes = axes.flatten()

for i, feat in enumerate(other_cols):
    ax = axes[i]
    doi_vals = pivot[(feat, 'DOI')].values
    wd_vals  = pivot[(feat, 'withdrawal')].values
    for d, w in zip(doi_vals, wd_vals):
        ax.plot([0, 1], [d, w], color='gray', alpha=0.5, linewidth=1)
    ax.scatter(np.zeros_like(doi_vals), doi_vals, color='crimson', zorder=3)
    ax.scatter(np.ones_like(wd_vals), wd_vals, color='steelblue', zorder=3)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['DOI', 'withdrawal'])
    p_val = results_df.loc[results_df['feature'] == feat, 'paired_t_p'].values[0]
    ax.set_title(f"{feat}\np={p_val:.3f}", fontsize=9)
    ax.set_xlim(-0.3, 1.3)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(output_folder / 'doi_vs_withdrawal_bout_features_paired.png', dpi=150)
plt.show()

In [ ]:
print(combined_2class_avg[['rat', 'condition', 'occ_0', 'n_bouts_0']])

In [ ]:
# 1) full occupancy across all clusters, per session (not just cluster 0)
full_occ_check = (
    df_full.groupby('json_name')['cluster']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)
full_occ_check['rat'] = full_occ_check.index.map(get_rat_id)
full_occ_check['condition'] = full_occ_check.index.map(get_condition)
print(full_occ_check.sort_values(['rat', 'condition']).to_string())

# 2) raw cluster sequence for one non-F6 session, e.g. F1 DOI
sample_session = 'F1DOITest_predictions.json'  # adjust if the exact name differs
sample_seq = (
    df_full[df_full['json_name'] == sample_session]
    .sort_values('start')['cluster']
    .values
)
print(f"\n{sample_session}: {len(sample_seq)} windows")
print("First 30 cluster labels:", sample_seq[:30])
print("Last 30 cluster labels:", sample_seq[-30:])
print("Unique clusters in this session:", np.unique(sample_seq))

In [ ]:
f1doi_latents = results['latents_2d'][f1doi['win_idx'].values]
f2doi_latents = results['latents_2d'][f2doi['win_idx'].values]

print("F1 DOI latents (should vary if raw movement varies):")
print(f1doi_latents[:10])
print(f"F1 DOI latent std: {f1doi_latents.std(axis=0)}")

print("\nF2 DOI latents:")
print(f2doi_latents[:10])
print(f"F2 DOI latent std: {f2doi_latents.std(axis=0)}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import numpy as np

# ── pick the rat and pull its three sessions ───────────────────────────────
target_rat = 'F7'   # adjust to your actual rat ID format, e.g. 'F1', 'Rat1', etc.

rat_sessions = df[df['rat'] == target_rat].copy()
print(rat_sessions['json_name'].unique())
print(rat_sessions.groupby('condition')['json_name'].unique())

# one session per condition — if a rat has multiple sessions per condition,
# adjust this to pick a specific json_name per condition instead
session_by_condition = {}
for cond in ['baseline', 'DOI', 'withdrawal']:
    matches = rat_sessions[rat_sessions['condition'] == cond]['json_name'].unique()
    if len(matches) == 0:
        print(f"  WARNING: no {cond} session found for rat {target_rat}")
        continue
    if len(matches) > 1:
        print(f"  NOTE: rat {target_rat} has {len(matches)} {cond} sessions, using first: {matches[0]}")
    session_by_condition[cond] = matches[0]

# ── build the ethogram: cluster sequence per window, in temporal order ────
# df was built from active['cluster_labels'] indexed by win_idx in order,
# so we need each session's windows in their original temporal order.
# Rebuild that here using win_idx explicitly.

window_json_full    = []
window_cluster_full = []
window_start_full   = []

for win_idx, cluster_label in enumerate(results['cluster_labels']):
    json_name, _ = frame_to_video[starts[win_idx]]
    window_json_full.append(json_name)
    window_cluster_full.append(cluster_label)
    window_start_full.append(starts[win_idx])

df_full = pd.DataFrame({
    'json_name': window_json_full,
    'cluster':   window_cluster_full,
    'start':     window_start_full,
})

# ── plot ────────────────────────────────────────────────────────────────
n_clusters = df_full['cluster'].nunique()
cluster_ids = sorted(df_full['cluster'].unique())
cmap = plt.get_cmap('tab10', n_clusters)

fig, axes = plt.subplots(len(session_by_condition), 1,
                          figsize=(14, 1.2 * len(session_by_condition) + 1),
                          sharex=False)
if len(session_by_condition) == 1:
    axes = [axes]

for ax, (cond, json_name) in zip(axes, session_by_condition.items()):
    sess = df_full[df_full['json_name'] == json_name].sort_values('start')
    clusters_seq = sess['cluster'].values
    n_windows = len(clusters_seq)

    # map cluster id -> color index
    color_idx = np.array([cluster_ids.index(c) for c in clusters_seq])
    img = color_idx.reshape(1, -1)

    ax.imshow(img, aspect='auto', cmap=cmap, vmin=0, vmax=n_clusters - 1,
              extent=[0, n_windows, 0, 1])
    ax.set_yticks([])
    ax.set_ylabel(cond, rotation=0, ha='right', va='center', fontsize=11)
    ax.set_xlim(0, n_windows)

axes[-1].set_xlabel('Window (time)')
fig.suptitle(f'Ethogram — Rat {target_rat}: cluster sequence by condition', y=1.02)

# shared legend
legend_patches = [mpatches.Patch(color=cmap(i), label=f'Cluster {cid}')
                   for i, cid in enumerate(cluster_ids)]
fig.legend(handles=legend_patches, loc='upper center',
           bbox_to_anchor=(0.5, 1.08), ncol=n_clusters, frameon=False)

plt.tight_layout()
plt.savefig(output_folder / f'ethogram_rat_{target_rat}.png', dpi=150, bbox_inches='tight')
plt.show()

# ── also print occupancy fractions for reference ───────────────────────────
print(f"\nCluster occupancy fractions — Rat {target_rat}:")
for cond, json_name in session_by_condition.items():
    occ = df_full[df_full['json_name'] == json_name]['cluster'].value_counts(normalize=True).sort_index()
    print(f"\n{cond} ({json_name}):")
    print(occ.round(3))

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# ── step: infer condition label from each window's video filename ─────────
CONDITION_PATTERNS = {
    'baseline':   re.compile(r'baseline', re.IGNORECASE),
    'DOI':        re.compile(r'doi', re.IGNORECASE),
    'withdrawal': re.compile(r'withdrawal', re.IGNORECASE),
}

def get_condition(json_name):
    for cond, pat in CONDITION_PATTERNS.items():
        if pat.search(json_name):
            return cond
    return 'unknown'

# build per-window condition + cluster label arrays
# (uses `frame_to_video`, `starts`, and `active['cluster_labels']` from your pipeline)
window_conditions = []
window_clusters   = []

for win_idx, cluster_label in enumerate(results['cluster_labels']):
    json_name, _ = frame_to_video[starts[win_idx]]
    cond = get_condition(json_name)
    window_conditions.append(cond)
    window_clusters.append(cluster_label)

window_conditions = np.array(window_conditions)
window_clusters   = np.array(window_clusters)

# optional: drop windows whose condition couldn't be determined
mask = window_conditions != 'unknown'
print(f"Dropping {(~mask).sum()} windows with unknown condition")
window_conditions = window_conditions[mask]
window_clusters   = window_clusters[mask]

# ── build the cross-tab (cluster rows x condition columns) ────────────────
cond_order = ['baseline', 'DOI', 'withdrawal']
cluster_order = sorted(np.unique(window_clusters))

ct = pd.crosstab(
    pd.Categorical(window_clusters, categories=cluster_order),
    pd.Categorical(window_conditions, categories=cond_order),
)
ct.index.name = 'Cluster'
ct.columns.name = 'Condition'

print(ct)

# ── plot as heatmap ─────────────────────────────────────────────────────
# raw counts
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(ct, annot=True, fmt='d', cmap=cmap, ax=axes[0], cbar=True)
axes[0].set_title('Cluster vs Condition (counts)')
axes[0].set_ylabel('Cluster')
axes[0].set_xlabel('Condition')

# row-normalized (fraction of each cluster's windows per condition)
ct_norm = ct.div(ct.sum(axis=1), axis=0)
sns.heatmap(ct_norm, annot=True, fmt='.2f', cmap=cmap, ax=axes[1], cbar=True)
axes[1].set_title('Cluster vs Condition (row-normalized)')
axes[1].set_ylabel('Cluster')
axes[1].set_xlabel('Condition')

plt.tight_layout()
plt.savefig(output_folder / 'cluster_vs_condition_confusion.png', dpi=150)
plt.show()

In [ ]:
results=new_best_results

In [ ]:
# ── group-centric composition: for each group, % of its windows in each cluster ──
labels      = results['cluster_labels']
n_clusters  = len(np.unique(labels))
group_order = ["baseline", "DOI", "withdrawal"]

rows = []
for g in group_order:
    group_mask = (window_groups == g)
    group_labels = labels[group_mask]
    n_total = len(group_labels)

    row = {"group": g, "n_windows": n_total}
    for c in range(n_clusters):
        n_c = np.sum(group_labels == c)
        pct = 100 * n_c / n_total if n_total > 0 else 0.0
        row[f"cluster_{c}_n"]   = n_c
        row[f"cluster_{c}_pct"] = round(pct, 1)
    rows.append(row)

group_composition_df = pd.DataFrame(rows)
print(group_composition_df.to_string(index=False))


# ── stacked bar chart: one bar per group, stacked by cluster ───────────────
import matplotlib.cm as cm

cluster_ids   = list(range(n_clusters))
cluster_cmap  = cm.get_cmap("tab10", n_clusters)
cluster_colors = {c: cluster_cmap(i) for i, c in enumerate(cluster_ids)}

groups = group_composition_df["group"].values
x      = np.arange(len(groups))

fig, ax = plt.subplots(figsize=(8, 5))

bottom = np.zeros(len(groups))
for c in cluster_ids:
    pct_col = f"cluster_{c}_pct"
    vals    = group_composition_df[pct_col].values
    ax.bar(x, vals, bottom=bottom, label=f"Cluster {c}",
           color=cluster_colors[c], edgecolor="white", linewidth=0.5)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels([g.replace("_", " ") for g in groups], rotation=20, ha="right")
ax.set_ylabel("% of windows")
ax.set_ylim(0, 100)
ax.set_title("Cluster Composition by Group")
ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
fig_path = output_folder / "group_cluster_composition.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved → {fig_path}")

In [ ]:
model_stage2 = HierarchicalRAE(
    latent_dim=30,
    joint_dim=8,
    num_joints=7,
    hidden_dim=256,
    pose_embed=128,
    joint_embed=32,
).to(device)
model_stage2.load_state_dict(torch.load(r"D:\_Dev\2026\vae_jackie\new_model_stage2.pth",map_location=device))
model_stage2.eval()
print("stage 2 model loaded")\


transition_frames, smoothed = find_transitions(
    positions,losses,percentile = 62.88, fps=30
)

windows,window_segment_labels = create_windows_from_transitions(
    raw_processed,transition_frames,
    min_segment_frames=60,max_segment_frames=600
)

all_latents = []
tensor_list=[torch.tensor(w,dtype=torch.float32) for w in windows]
loader = torch.utils.data.DataLoader(
    tensor_list,batch_size=BATCH_SIZE,shuffle=False,
    collate_fn=collate_variable_length
)

with torch.no_grad():
    for padded,lengths in loader:
        padded=padded.to(device)
        _,z = model_stage2(padded,lengths=lengths)
        all_latents.append(z.cpu().numpy())
all_latents = np.concatenate(all_latents,axis=0)
print(f"latents shape: {all_latents.shape}")

# Stage 5: encode + cluster
latents, latents_2d, cluster_labels, ms_model = encode_and_cluster(
    model_stage2, windows, quantile=0.1395,
    umap_neighbors=30, umap_min_dist=0.1, device=device
)

n_clusters = len(np.unique(cluster_labels))

    # ── Compute silhouette score ──────────────────────────────
    # silhouette needs at least 2 clusters and more samples than clusters
if n_clusters < 2 or n_clusters >= len(latents_2d):
    print(f"  → {n_clusters} clusters — skipping silhouette (invalid cluster count)")
    score = -1.0
else:
    score = silhouette_score(latents_2d, cluster_labels)

print(f"found {len(np.unique(cluster_labels))} clusters")
print(f"Cluster sizes: {np.bincount(cluster_labels)}")


results = {
    'model': model_stage2,
    'latents': all_latents,
    'latents_2d': latents_2d,
    'cluster_labels': cluster_labels,
    'transition_frames': transition_frames,
    'windows': windows,
    'window_segment_labels': window_segment_labels,
}

In [ ]:
print(bandwidth)

In [ ]:
import cv2
import json
import numpy as np
from pathlib import Path
from collections import defaultdict

# ── settings ───────────────────────────────────────────────────────────────
json_folder   = Path(r"E:\ferg_take2\json_files")
video_folder  = Path(r"E:\ferg_take2\videos")
output_folder = Path(r"E:\ferg_take2\reconstructed_vids")
output_folder.mkdir(exist_ok=True)

n_cols             = 2
n_rows             = 2
n_examples         = n_cols * n_rows   # 12
fps_out            = 30
cell_w, cell_h     = 320, 240
min_segment_frames = 60
max_segment_frames = 600

# ── step 1: read in every JSON file present, no hardcoded filenames ───────
# IMPORTANT: order must exactly match how `da` was originally built, which
# used unsorted `list(json_folder.glob("*.json"))` (NOT alphabetically
# sorted) — so this uses the identical unsorted glob call. glob() order is
# just OS directory-listing order; it's consistent across runs as long as
# no files in json_folder have been added/removed/renamed since `da` was
# built. If you're not certain that's true, don't trust this without
# spot-checking (see the verification block below).
json_files_found = list(json_folder.glob("*.json"))
print(f"Found {len(json_files_found)} JSON files in {json_folder}")

print("\nLoading JSON files...")
json_names   = []
frame_counts = []

for json_file in json_files_found:
    json_name = json_file.name
    with open(json_file) as f:
        data = json.load(f)
    n = len(data["annotations"])
    json_names.append(json_name)
    frame_counts.append(n)
    print(f"  {json_name}: {n} frames")

json_names_ordered = json_names  # kept for compatibility with steps below

print(f"\nTotal frames: {sum(frame_counts)}")
assert sum(frame_counts) == len(da), \
    f"Frame count mismatch: {sum(frame_counts)} vs {len(da)}"
print("Frame counts match da ✓")

# ── step 2: build json → actual video frame ID mapping ────────────────────
print("\nBuilding frame ID maps...")
json_frame_maps = {}
for json_name in json_names_ordered:
    json_file = json_folder / json_name
    with open(json_file) as f:
        data = json.load(f)
    frame_ids = list(data["annotations"].keys())
    json_frame_maps[json_name] = {
        i: int(fid) for i, fid in enumerate(frame_ids)
    }
    print(f"  {json_name}: frames {frame_ids[0]} → {frame_ids[-1]}")

# ── step 3: build global frame → (json_name, local_annotation_idx) ────────
print("\nBuilding frame_to_video mapping...")
frame_to_video = {}
cumulative     = 0
for name, count in zip(json_names, frame_counts):
    for local_frame in range(count):
        frame_to_video[cumulative + local_frame] = (name, local_frame)
    cumulative += count
print(f"frame_to_video built: {len(frame_to_video)} entries")

# ── verify mapping is correct ──────────────────────────────────────────────
# NOTE: this only checks index 0. Given the ordering caveat above, it's
# worth manually checking one or two more indices (e.g. the first frame of
# the 2nd and last json files) before trusting the full mapping.
json_name_0, local_idx_0 = frame_to_video[0]
actual_vid_0 = json_frame_maps[json_name_0].get(local_idx_0, local_idx_0)
json_file_0  = json_folder / json_name_0
with open(json_file_0) as f:
    d0 = json.load(f)
frame_ids_0 = list(d0["annotations"].keys())
first_kps   = list(d0["annotations"][frame_ids_0[0]].values())
first_x     = [kp[0] for kp in first_kps]
da_x        = da.values[0, :, 0].tolist()
match       = np.allclose(first_x, da_x, atol=1)
print(f"\nVerification: frame_to_video[0] → {json_name_0}")
print(f"  json x: {first_x}")
print(f"  da x:   {da_x}")
print(f"  Match: {match} {'✓' if match else '✗ — ORDER IS WRONG'}")

# ── step 4: find matching video files (match by stem, minus '_predictions') ─
print("\nSearching for video files...")
all_videos    = list(video_folder.rglob("*.avi"))
video_by_name = {v.stem: v for v in all_videos}

video_map = {}
for json_name in json_names:
    stem = json_name.replace('_predictions.json', '')
    if stem in video_by_name:
        video_map[json_name] = video_by_name[stem]
        print(f"  ✓ {json_name}")
    else:
        print(f"  ✗ {json_name} — no matching video")
print(f"video_map has {len(video_map)} entries")

In [ ]:
# ── step 5: reconstruct window start frames ────────────────────────────────
print("\nReconstructing window starts...")

# use the best iteration from the Bayesian search over balanced windows,
# not the original `results` — its windows/clusters are stale for this run
active = best_results

n_frames_total = len(da)
boundaries     = np.unique(
    np.concatenate([[0], results['transition_frames'], [n_frames_total]])
).astype(int)

starts = []
for seg_idx in range(len(boundaries) - 1):
    seg_start = boundaries[seg_idx]
    seg_end   = boundaries[seg_idx + 1]
    seg_len   = seg_end - seg_start
    if seg_len < min_segment_frames:
        continue
    for chunk_start in range(0, seg_len, max_segment_frames):
        abs_start = seg_start + chunk_start
        abs_end   = min(abs_start + max_segment_frames, seg_end)
        if (abs_end - abs_start) < min_segment_frames:
            continue
        starts.append(abs_start)

starts = np.array(starts)
print(f"Window starts: {len(starts)} vs windows: {len(results['windows'])}")
assert len(starts) == len(results['windows']), \
    f"Mismatch: {len(starts)} starts vs {len(results['windows'])} windows"


In [ ]:
# ============================================================
# Visual representation of the significant motifs — pose only
# Reuses: da, results, starts, frame_to_video, video_map, json_names_ordered
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

START_STOP = LinearSegmentedColormap.from_list(
    "start_stop", ["#2ab7e0", "#ffffff", "#d6418f"])

SKELETON    = [(0,1),(1,2),(1,3),(1,4),(2,5),(2,6)]
NOSE        = 0
MAX_FRAMES  = 80       # max poses to overlay per panel
ALPHA_SKEL  = 0.25     # skeleton stroke opacity
ALPHA_NOSE  = 0.9      # nose trajectory opacity
PAD         = 0.15     # axis padding (fraction)
MOTIF_NAMES = {}       # optional {0: "Turning", ...}

labels  = results['cluster_labels']
latents = results['latents']           # full-dim for accurate centroid distance


def window_has_video(w):
    """A window is usable only if every frame in it:
      1. maps to a json/video session (frame_to_video), and
      2. that session actually has a matching video file (video_map), and
      3. the window doesn't straddle two different sessions (which would
         make the pose sequence jump between unrelated videos).
    """
    abs_start = starts[w]
    L         = len(results['windows'][w])
    abs_end   = abs_start + L - 1

    if abs_start not in frame_to_video or abs_end not in frame_to_video:
        return False

    start_json, _ = frame_to_video[abs_start]
    end_json, _   = frame_to_video[abs_end]

    if start_json != end_json:
        return False  # window spans two sessions — skip it

    return start_json in video_map


def representative_window(c):
    idx = np.where(labels == c)[0]
    if len(idx) == 0:
        return None
    cen   = latents[idx].mean(axis=0)
    order = idx[np.argsort(np.linalg.norm(latents[idx] - cen, axis=1))]
    for w in order:
        if window_has_video(int(w)):
            return int(w)
    return None


def draw_motif_pose(ax, win_idx):
    abs_start = starts[win_idx]
    L         = len(results['windows'][win_idx])

    # evenly subsample up to MAX_FRAMES poses
    step = max(1, L // MAX_FRAMES)
    kp   = da.values[abs_start:abs_start + L:step][:MAX_FRAMES]  # (T, 7, 2)
    T    = len(kp)

    ax.set_facecolor("black")

    for i in range(T):
        t   = i / max(T - 1, 1)
        col = START_STOP(t)

        # skeleton limbs
        for j0, j1 in SKELETON:
            xs = [kp[i, j0, 0], kp[i, j1, 0]]
            ys = [kp[i, j0, 1], kp[i, j1, 1]]
            ax.plot(xs, ys, color=col, lw=1.8, alpha=ALPHA_SKEL,
                    solid_capstyle="round")

        # joint dots
        ax.scatter(kp[i, :, 0], kp[i, :, 1],
                   s=12, color=col, alpha=ALPHA_SKEL, zorder=3, edgecolors="none")

    # nose trajectory on top as white dotted line
    nose = kp[:, NOSE, :]
    ax.plot(nose[:, 0], nose[:, 1], ls=":", color="white", lw=1.4,
            marker="o", ms=2, mfc="white", mec="white", alpha=ALPHA_NOSE, zorder=5)

    # auto-scale with padding
    all_xy = kp.reshape(-1, 2)
    xmin, ymin = all_xy.min(0); xmax, ymax = all_xy.max(0)
    xpad = PAD * (xmax - xmin); ypad = PAD * (ymax - ymin)
    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymax + ypad, ymin - ypad)   # invert y (image coords)
    ax.set_xticks([]); ax.set_yticks([])


# ---- assemble figure ----
n_clusters = len(np.unique(labels))
panels = []
for c in range(n_clusters):
    w = representative_window(c)
    if w is not None:
        panels.append((c, w))
    else:
        print(f"  ✗ Cluster {c}: no window with a usable video found — skipping panel")

ncol = min(4, len(panels))
nrow = int(np.ceil(len(panels) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.4 * nrow), facecolor="white")
axes = np.atleast_1d(axes).ravel()
for ax in axes[len(panels):]:
    ax.axis("off")

for k, (ax, (c, w)) in enumerate(zip(axes, panels)):
    draw_motif_pose(ax, w)
    ax.text(0.03, 0.97, MOTIF_NAMES.get(c, f"Motif {c}"), transform=ax.transAxes,
            color="white", fontsize=11, va="top")
    ax.text(0.97, 0.97, str(c), transform=ax.transAxes, color="white",
            fontsize=20, fontweight="bold", va="top", ha="right")
    if k == 0:
        cax = ax.inset_axes([0.28, 0.04, 0.44, 0.05])
        cax.imshow(np.linspace(0, 1, 256)[None, :], aspect="auto", cmap=START_STOP)
        cax.set_xticks([]); cax.set_yticks([])
        ax.text(0.28, 0.11, "Start", transform=ax.transAxes, color="#2ab7e0", fontsize=9)
        ax.text(0.72, 0.11, "Stop",  transform=ax.transAxes, color="#d6418f",
                fontsize=9, ha="right")

fig.suptitle("Visual representation of the significant motifs", fontsize=14)
plt.tight_layout()
fig.savefig(output_folder / "significant_motifs_pose.png", dpi=200,
            bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# quick check before running:
for c in range(len(np.unique(results['cluster_labels']))):
    w = representative_window(c)
    if w is None: continue
    abs_start = starts[w]
    L = len(results['windows'][w])
    burst_start = abs_start + max(0, (L - 60) // 2)
    burst_end = burst_start + 60
    sess_start = frame_to_video[burst_start][0]
    sess_end   = frame_to_video[min(burst_end, len(frame_to_video)-1)][0]
    print(f"cluster {c}: window {w}, session {sess_start} -> {sess_end}, same={sess_start==sess_end}")

In [ ]:
new_best_results = results
active=results

In [ ]:
# ── step 5b: build the mirror lookup ────────────────────────────────────────
# Mirror folder files are named exactly like the json files themselves
# (e.g. "F2DOITest_predictions.json"), so match on full filename, not stem.
mirror_folder = Path(r"E:\ferg_take2\mirror2")

mirror_names = {p.name for p in mirror_folder.glob("*")} if mirror_folder.exists() else set()
print(f"\nFound {len(mirror_names)} files in mirror folder: {mirror_folder}")

# map each json_name -> whether its matching video should be mirrored
should_mirror = {json_name: json_name in mirror_names for json_name in json_names}

n_mirrored = sum(should_mirror.values())
print(f"{n_mirrored} / {len(json_names)} videos will be mirrored")
if n_mirrored:
    mirrored_names = [n for n, m in should_mirror.items() if m]
    print("  " + "\n  ".join(mirrored_names))


# ── step 6: skeleton and helper functions ──────────────────────────────────
SKELETON = [(0,1),(1,2),(1,3),(1,4),(2,5),(2,6)]

def draw_skeleton(frame, keypoints, color=(0,255,0), thickness=2):
    kp = keypoints.astype(int)
    for (a, b) in SKELETON:
        cv2.line(frame, tuple(kp[a]), tuple(kp[b]), color, thickness)
    for (x, y) in kp:
        cv2.circle(frame, (x, y), 4, color, -1)
    return frame

_cap_cache = {}
_cap_pos   = {}

def get_frame_sequential(json_name, local_frm):
    if json_name not in video_map:
        return None
    actual_frm = json_frame_maps[json_name].get(local_frm, local_frm)

    if json_name not in _cap_cache:
        _cap_cache[json_name] = cv2.VideoCapture(str(video_map[json_name]))
        _cap_pos[json_name]   = 0

    cap = _cap_cache[json_name]
    cur = _cap_pos[json_name]

    if actual_frm < cur:
        cap.release()
        _cap_cache[json_name] = cv2.VideoCapture(str(video_map[json_name]))
        _cap_pos[json_name]   = 0
        cap = _cap_cache[json_name]
        cur = 0

    while cur < actual_frm:
        cap.read()
        cur += 1

    ret, frame = cap.read()
    cur += 1
    _cap_pos[json_name] = cur
    return frame if ret else None

def close_cap_cache():
    for cap in _cap_cache.values():
        cap.release()
    _cap_cache.clear()
    _cap_pos.clear()

def window_has_video(win_idx):
    window    = active['windows'][win_idx]
    abs_start = starts[win_idx]
    for local_t in range(min(5, len(window))):
        global_frame  = abs_start + local_t
        json_name, _  = frame_to_video[global_frame]
        if json_name not in video_map:
            return False
    return True

def load_window_frames(win_idx):
    window    = active['windows'][win_idx]
    abs_start = starts[win_idx]
    frames    = []
    for local_t in range(len(window)):
        global_frame         = abs_start + local_t
        json_name, local_frm = frame_to_video[global_frame]
        frame = get_frame_sequential(json_name, local_frm)
        if frame is not None:
            raw_kp = da.values[global_frame]

            # mirror the raw video frame — only for videos flagged in the
            # mirror lookup — keypoints are left completely unchanged
            if should_mirror.get(json_name, False):
                frame = cv2.flip(frame, 1)

            frame = draw_skeleton(frame, raw_kp, color=(0, 255, 0))
            frame = cv2.resize(frame, (cell_w, cell_h))
        else:
            frame  = np.zeros((cell_h, cell_w, 3), dtype=np.uint8)
        frames.append(frame)
    return frames

# ── step 7: create one grid video per cluster ──────────────────────────────
n_clusters = len(np.unique(results['cluster_labels']))
latents_2d = active['latents_2d']
labels     = active['cluster_labels']

print(f"\nCreating grid videos for {n_clusters} clusters (from new_best_results)...")

for c in range(n_clusters):
    print(f"\nCluster {c}...")

    cluster_idx     = np.where(labels == c)[0]
    cluster_latents = latents_2d[cluster_idx]
    centroid        = cluster_latents.mean(axis=0)
    dists           = np.linalg.norm(cluster_latents - centroid, axis=1)
    closest_order   = np.argsort(dists)

    chosen = []
    for idx in cluster_idx[closest_order]:
        if window_has_video(idx):
            chosen.append(idx)
        if len(chosen) == n_examples:
            break
    chosen = np.array(chosen)
    print(f"  {len(cluster_idx)} windows total, {len(chosen)} with valid video")

    if len(chosen) == 0:
        print(f"  Skipping cluster {c} — no valid videos")
        continue

    close_cap_cache()

    print(f"  Loading frames...")
    all_window_frames = []
    for i, win_idx in enumerate(chosen):
        print(f"    Window {i+1}/{len(chosen)} "
              f"(idx={win_idx}, len={len(active['windows'][win_idx])})")
        wf = load_window_frames(win_idx)
        all_window_frames.append(wf)

    max_len  = max(len(wf) for wf in all_window_frames)
    grid_w   = cell_w * n_cols
    grid_h   = cell_h * n_rows + 40
    out_path = output_folder / f"cluster_{c}_grid.mp4"
    fourcc   = cv2.VideoWriter_fourcc(*'mp4v')
    out      = cv2.VideoWriter(str(out_path), fourcc, fps_out, (grid_w, grid_h))
    blank    = np.zeros((cell_h, cell_w, 3), dtype=np.uint8)

    print(f"  Writing {max_len} frames to {out_path.name}...")
    for t in range(max_len):
        grid = np.zeros((grid_h, grid_w, 3), dtype=np.uint8)
        cv2.putText(grid, f"Cluster {c}  |  Frame {t+1}/{max_len}",
                    (10, 28), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (255, 255, 255), 2)

        for i in range(n_examples):
            row     = i // n_cols
            col     = i % n_cols
            y_start = 40 + row * cell_h
            x_start = col * cell_w

            if i < len(all_window_frames):
                wf    = all_window_frames[i]
                frame = wf[t] if t < len(wf) else blank.copy()
            else:
                frame = blank.copy()

            if frame is None:
                frame = blank.copy()

            label_text = f"W{chosen[i]}"
            json_name_i, _ = frame_to_video[starts[chosen[i]]]
            if should_mirror.get(json_name_i, False):
                label_text += " (mirrored)"

            cv2.putText(frame, label_text,
                        (5, 20), cv2.FONT_HERSHEY_SIMPLEX,
                        0.5, (0, 255, 0), 1)

            grid[y_start:y_start+cell_h, x_start:x_start+cell_w] = frame

        out.write(grid)

    out.release()
    print(f"  Saved → {out_path}")

close_cap_cache()
print(f"\nDone! {n_clusters} videos saved to {output_folder}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── get centroid position (mean x, y of keypoints) for each window ────────
raw_centroids = da.values.mean(axis=1)  # shape: (n_frames, 2) -> mean over the 7 keypoints

window_centroids = np.array([raw_centroids[starts[i]] for i in range(len(active['cluster_labels']))])
labels = active['cluster_labels']

unique_labels = np.sort(np.unique(labels))
n_clusters = len(unique_labels)

# ── one scatter plot per cluster, all on the same axes for comparison ────
fig, axes = plt.subplots(1, n_clusters, figsize=(5 * n_clusters, 5), sharex=True, sharey=True)
if n_clusters == 1:
    axes = [axes]

for ax, c in zip(axes, unique_labels):
    mask = labels == c
    ax.scatter(window_centroids[mask, 0], window_centroids[mask, 1], s=8, alpha=0.4, color='steelblue')
    ax.set_title(f'Cluster {c} (n={mask.sum()})')
    ax.set_xlabel('X position')
    ax.invert_yaxis()  # flip y-axis so it matches typical video coordinate orientation (0,0 top-left)

axes[0].set_ylabel('Y position')
plt.suptitle('Rat arena position by cluster')
plt.tight_layout()
plt.savefig(output_folder / 'arena_position_by_cluster.png', dpi=150)
plt.show()

# ── overlaid version: all clusters on one plot, colored ────────────────────
plt.figure(figsize=(8, 7))
cmap = plt.get_cmap('tab10', n_clusters)
for i, c in enumerate(unique_labels):
    mask = labels == c
    plt.scatter(window_centroids[mask, 0], window_centroids[mask, 1],
                s=8, alpha=0.4, color=cmap(i), label=f'Cluster {c}')
plt.gca().invert_yaxis()
plt.xlabel('X position')
plt.ylabel('Y position')
plt.title('Rat arena position, colored by cluster')
plt.legend(markerscale=3)
plt.tight_layout()
plt.savefig(output_folder / 'arena_position_by_cluster_overlay.png', dpi=150)
plt.show()

In [ ]:
win_idx = 1119

abs_start = starts[win_idx]
json_name, local_frm = frame_to_video[abs_start]

print(f"Window {win_idx}:")
print(f"  abs_start frame: {abs_start}")
print(f"  source json:     {json_name}")
print(f"  local frame idx: {local_frm}")
print(f"  window length:   {len(active['windows'][win_idx])} frames")
print(f"  cluster:         {active['cluster_labels'][win_idx]}")
print(f"  mirrored:        {should_mirror.get(json_name, False)}")
print(f"  has video:       {json_name in video_map}")
if json_name in video_map:
    print(f"  video file:      {video_map[json_name]}")

In [ ]:
"""
plt.plot(best_results['lossgraph_stage1'], label='stage 1')
plt.plot(best_results['lossgraph_stage2'], label='stage 2')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()
"""

Save best_results to local disk

In [ ]:
with open("best_results.pkl", "wb") as file:
    pickle.dump(results, file)

Load best_results from local disk

In [ ]:
with open("best_results.pkl", "rb") as file:
    results = pickle.load(file)

#### Bayesian search of parameters with balance windows

In [21]:
# ── Step 1: balance windows based on EXISTING cluster labels ───────────────
print("="*60)
print("STEP 1: Balancing windows based on existing clusters")
print("="*60)

cluster_labels_original = results['cluster_labels']
cluster_sizes_original  = np.bincount(cluster_labels_original)
print(f"Original cluster sizes: {cluster_sizes_original}")
print(f"Total windows: {len(cluster_labels_original)}")

target_size = int(np.median(cluster_sizes_original))
print(f"Target size per cluster: {target_size}")

np.random.seed(42)
balanced_indices = []
for c in np.unique(cluster_labels_original):
    idx    = np.where(cluster_labels_original == c)[0]
    chosen = np.random.choice(idx, min(target_size, len(idx)), replace=False)
    balanced_indices.extend(chosen)
    print(f"  Cluster {c}: {len(idx):5d} → {min(target_size, len(idx)):5d} windows")

balanced_indices = np.array(balanced_indices)
balanced_windows = [results['windows'][i] for i in balanced_indices]
print(f"\nBalanced dataset: {len(balanced_windows)} windows total")



STEP 1: Balancing windows based on existing clusters
Original cluster sizes: [1283  285  258  203  192]
Total windows: 2221
Target size per cluster: 258
  Cluster 0:  1283 →   258 windows
  Cluster 1:   285 →   258 windows
  Cluster 2:   258 →   258 windows
  Cluster 3:   203 →   203 windows
  Cluster 4:   192 →   192 windows

Balanced dataset: 1169 windows total


In [22]:
# ── Step 2: compute transition frames from existing results ────────────────
# these are reused inside the Bayesian loop for percentile tuning
positions        = results['positions']
losses           = results['losses']
all_windows_orig = results['windows']   # full unbalanced window set


In [23]:
# ── Step 3: Bayesian optimization ─────────────────────────────────────────
print("\n" + "="*60)
print("STEP 3: Bayesian optimization over lr, quantile, percentile")
print("="*60)

MAX_ITER  = 10
VW_EPOCHS = 750

search_space = [
    Real(*PERCENTILE_RANGE, name='percentile'),
    Real(*QUANTILE_RANGE, name='quantile'),
    #Real(*LR_RANGE, name='lr', prior='log-uniform'),
]

new_search_log   = []
new_best_results = None
new_best_score   = -1
new_iteration    = 0



STEP 3: Bayesian optimization over lr, quantile, percentile


In [24]:
@use_named_args(search_space)
def new_objective(percentile, quantile, lr=LR):
    global new_iteration, new_best_score, new_best_results

    new_iteration += 1
    print(f"\n{'='*60}")
    print(f"ITERATION {new_iteration}/{MAX_ITER} | percentile={percentile:.2f}, quantile={quantile:.4f}, lr={lr:.6f}")
    print(f"{'='*60}")

    # ── find new transitions with this percentile ──────────────────────────
    new_transition_frames, new_smoothed = find_transitions(
        positions, losses, percentile=percentile, fps=30
    )

    # ── create new variable windows with these transitions ─────────────────
    new_windows, new_window_segment_labels = create_windows_from_transitions(
        raw_processed, new_transition_frames,
        min_segment_frames=60, max_segment_frames=600
    )

    if len(new_windows) < 10:
        print(f"  → too few windows ({len(new_windows)}) — skipping")
        return 0.0

    # ── map balanced indices to new windows ────────────────────────────────
    # balanced_indices came from the original window set
    # we need to find the corresponding windows in the new set
    # use the balanced windows directly since they came from raw_processed
    # and are independent of transition detection
    train_windows = balanced_windows   # always train on balanced subset

    print(f"  Training on {len(train_windows)} balanced windows...")

    # ── retrain Stage 2 on balanced windows with this lr ──────────────────
    new_model_file_name = "new_model_stage2_"+str(new_iteration)+".pth"
    if Path(new_model_file_name).is_file():
        n_joints  = raw_processed.shape[1]
        joint_dim = raw_processed.shape[2]

        new_model_stage2 = HierarchicalRAE(latent_dim=30, joint_dim=joint_dim,
                                num_joints=n_joints).to(device)
        state_dict = torch.load(new_model_file_name, weights_only=True)
        new_model_stage2.load_state_dict(state_dict)
        print("new_model_satge2 loaded...")
    else:
        new_model_stage2, new_lossgraph_stage2 = train_on_variable_windows(
            raw_processed, train_windows, epochs = VW_EPOCHS, lr = lr, device = device
        )
        torch.save(new_model_stage2.state_dict(), os.path.join('./', new_model_file_name))

    # ── encode ALL new windows with trained model ──────────────────────────
    print(f"  Encoding {len(new_windows)} windows...")
    new_latents, new_latents_2d, new_cluster_labels, new_ms_model = encode_and_cluster(
        new_model_stage2, new_windows,   # ALL new windows
        quantile = quantile, umap_neighbors = 30, umap_min_dist = 0.1, device = device
    )

    new_n_clusters = len(np.unique(new_cluster_labels))

    # ── silhouette score ───────────────────────────────────────────────────
    if new_n_clusters < 2 or new_n_clusters >= len(new_latents_2d):
        print(f"  → {new_n_clusters} clusters — invalid")
        new_score = -1.0
    else:
        normed      = normalize(new_latents_2d, norm='l2')
        dist_matrix = np.clip(1 - np.dot(normed, normed.T), 0, 2)
        new_score       = silhouette_score(dist_matrix, new_cluster_labels,
                                        metric='precomputed')

    new_log_entry = {
        "iteration":  new_iteration,
        "percentile": round(percentile, 2),
        "quantile":   round(quantile, 4),
        "lr":         round(float(lr), 6),
        "n_clusters": new_n_clusters,
        "silhouette": round(float(new_score), 4),
    }
    new_search_log.append(new_log_entry)
    print(f"  → {new_n_clusters} clusters, silhouette={new_score:.4f}")

    if new_score > new_best_score:
        new_best_score = new_score
        new_best_results = {
            'model':                 new_model_stage2,
            'model_stage1':          results['model_stage1'],
            'latents':               new_latents,
            'latents_2d':            new_latents_2d,
            'cluster_labels':        new_cluster_labels,
            'transition_frames':     new_transition_frames,
            'windows':               new_windows,
            'window_segment_labels': new_window_segment_labels,
            'losses':                losses,
            'positions':             positions,
            'lossgraph_stage1':      results['lossgraph_stage1'],
            'lossgraph_stage2':      new_lossgraph_stage2,
            'smoothed_losses':       new_smoothed,
            '_percentile':           percentile,
            '_quantile':             quantile,
            '_lr':                   lr,
        }
        print(f"  ** NEW BEST (silhouette={new_score:.4f}) **")

    del new_model_stage2, new_latents, new_latents_2d, new_cluster_labels, new_ms_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return -new_score

In [25]:
# ── run optimization ───────────────────────────────────────────────────────
new_bayes_result = gp_minimize(
    func             = new_objective,
    dimensions       = search_space,
    n_calls          = MAX_ITER,
    n_initial_points = min(MAX_ITER, 5),
    random_state     = 42,
    verbose          = False,
)




ITERATION 1/10 | percentile=63.81, quantile=0.0775, lr=0.001000
Found 6180 transitions
Mean bout duration: 2.41s
Bout duration looks plausible
Created 2146 variable-length windows from 6181 segments
Window lengths — min: 60, max: 600, mean: 155.0
  Training on 1169 balanced windows...
Retraining on variable-length windows...


c:\Users\fbai_\anaconda3\envs\condaenv312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 10/750 — Loss: 0.327262  patience: 0/30
Epoch 20/750 — Loss: 0.296878  patience: 0/30
Epoch 30/750 — Loss: 0.287730  patience: 2/30
Epoch 40/750 — Loss: 0.280491  patience: 1/30
Epoch 50/750 — Loss: 0.274059  patience: 4/30
Epoch 60/750 — Loss: 0.269512  patience: 4/30
Epoch 70/750 — Loss: 0.263966  patience: 0/30
Epoch 80/750 — Loss: 0.267825  patience: 10/30
Epoch 90/750 — Loss: 0.261945  patience: 1/30
Epoch 100/750 — Loss: 0.260689  patience: 11/30
Epoch 110/750 — Loss: 0.256060  patience: 4/30
Epoch 120/750 — Loss: 0.253538  patience: 14/30
Epoch 130/750 — Loss: 0.252492  patience: 2/30
Epoch 140/750 — Loss: 0.252684  patience: 12/30
Epoch 150/750 — Loss: 0.251703  patience: 2/30
Epoch 160/750 — Loss: 0.247490  patience: 6/30
Epoch 170/750 — Loss: 0.247781  patience: 4/30
Epoch 180/750 — Loss: 0.245164  patience: 0/30
Epoch 190/750 — Loss: 0.250564  patience: 10/30
Epoch 200/750 — Loss: 0.253174  patience: 20/30
Early stopping at epoch 210 — best loss: 0.245164
  Encoding 21

c:\Users\fbai_\anaconda3\envs\condaenv312\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP done. Shape: (2146, 2)
Estimated bandwidth: 0.0579
Found 9 clusters
Cluster sizes: [423 409 272 420 190 149 105  87  91]
Silhouette score: 0.7664
  → 9 clusters, silhouette=0.7664
  ** NEW BEST (silhouette=0.7664) **

ITERATION 2/10 | percentile=62.88, quantile=0.1395, lr=0.001000
Found 6304 transitions
Mean bout duration: 2.36s
Bout duration looks plausible
Created 2156 variable-length windows from 6305 segments
Window lengths — min: 60, max: 600, mean: 152.7
  Training on 1169 balanced windows...
Retraining on variable-length windows...


...

UMAP done. Shape: (1964, 2)
Estimated bandwidth: 0.3390
Found 5 clusters
Cluster sizes: [813 420 407 261  63]
Silhouette score: 0.7753
  → 5 clusters, silhouette=0.7753

ITERATION 10/10 | percentile=65.49, quantile=0.1464, lr=0.001000
Found 5985 transitions
Mean bout duration: 2.49s
Bout duration looks plausible
Created 2109 variable-length windows from 5986 segments
Window lengths — min: 60, max: 600, mean: 159.9
  Training on 1169 balanced windows...
Retraining on variable-length windows...


c:\Users\fbai_\anaconda3\envs\condaenv312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 10/750 — Loss: 0.320617  patience: 0/30
Epoch 20/750 — Loss: 0.295472  patience: 0/30
Epoch 30/750 — Loss: 0.288705  patience: 1/30
Epoch 40/750 — Loss: 0.278562  patience: 1/30
Epoch 50/750 — Loss: 0.277673  patience: 5/30
Epoch 60/750 — Loss: 0.268523  patience: 5/30
Epoch 70/750 — Loss: 0.265265  patience: 1/30
Epoch 80/750 — Loss: 0.268434  patience: 2/30
Epoch 90/750 — Loss: 0.259238  patience: 2/30
Epoch 100/750 — Loss: 0.274494  patience: 9/30
Epoch 110/750 — Loss: 0.253886  patience: 0/30
Epoch 120/750 — Loss: 0.247978  patience: 0/30
Epoch 130/750 — Loss: 0.251481  patience: 7/30
Epoch 140/750 — Loss: 0.243215  patience: 0/30
Epoch 150/750 — Loss: 0.245172  patience: 10/30
Epoch 160/750 — Loss: 0.247030  patience: 20/30
Epoch 170/750 — Loss: 0.245764  patience: 4/30
Epoch 180/750 — Loss: 0.239253  patience: 0/30
Epoch 190/750 — Loss: 0.241036  patience: 4/30
Epoch 200/750 — Loss: 0.242995  patience: 14/30
Epoch 210/750 — Loss: 0.242957  patience: 24/30
Early stopping at 

c:\Users\fbai_\anaconda3\envs\condaenv312\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP done. Shape: (2109, 2)
Estimated bandwidth: 0.1417
Found 7 clusters
Cluster sizes: [643 599 257 256 120 128 106]
Silhouette score: 0.7895
  → 7 clusters, silhouette=0.7895


In [26]:
# ── summary ────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE")
print(f"{'='*60}")
print(f"\nSearch log:")
for entry in new_search_log:
    marker = " <-- BEST" if entry['silhouette'] == new_best_score else ""
    print(f"  Iter {entry['iteration']:2d}: "
          f"percentile={entry['percentile']:5.2f}, "
          f"quantile={entry['quantile']:.4f}, "
          f"lr={entry['lr']:.6f}  "
          f"→  {entry['n_clusters']} clusters, "
          f"silhouette={entry['silhouette']:.4f}{marker}")

print(f"\nBest parameters:")
print(f"  percentile:  {new_best_results['_percentile']:.2f}")
print(f"  quantile:    {new_best_results['_quantile']:.4f}")
print(f"  lr:          {new_best_results['_lr']:.6f}")
print(f"  n_clusters:  {len(np.unique(new_best_results['cluster_labels']))}")
print(f"  silhouette:  {new_best_score:.4f}")

# ── set results to best ────────────────────────────────────────────────────
new_results = new_best_results

# Save the model
torch.save(new_results['model'].state_dict(), os.path.join('./', "new_model_stage2.pth"))

with open("new_lossgraph_stage2.json", "w") as file:
    json.dump(new_results['lossgraph_stage2'], file)


OPTIMIZATION COMPLETE

Search log:
  Iter  1: percentile=63.81, quantile=0.0775, lr=0.001000  →  9 clusters, silhouette=0.7664
  Iter  2: percentile=62.88, quantile=0.1395, lr=0.001000  →  7 clusters, silhouette=0.7905
  Iter  3: percentile=44.52, quantile=0.0650, lr=0.001000  →  9 clusters, silhouette=0.8127
  Iter  4: percentile=45.26, quantile=0.1001, lr=0.001000  →  9 clusters, silhouette=0.7496
  Iter  5: percentile=27.86, quantile=0.1476, lr=0.001000  →  5 clusters, silhouette=0.8063
  Iter  6: percentile=47.46, quantile=0.0625, lr=0.001000  →  11 clusters, silhouette=0.7878
  Iter  7: percentile=44.50, quantile=0.0665, lr=0.001000  →  12 clusters, silhouette=0.7594
  Iter  8: percentile=69.02, quantile=0.1463, lr=0.001000  →  4 clusters, silhouette=0.8328
  Iter  9: percentile=72.86, quantile=0.1576, lr=0.001000  →  5 clusters, silhouette=0.7753
  Iter 10: percentile=65.49, quantile=0.1464, lr=0.001000  →  7 clusters, silhouette=0.7895

Best parameters:
  percentile:  69.02
  q

In [ ]:
with open("new_best_results.pkl", "wb") as file:
    pickle.dump(new_results, file)

In [ ]:
embedded = new_results['latents']
labels = new_results['cluster_labels']
plt.figure(figsize=(8, 6))
plt.scatter(embedded[:, 0], embedded[:, 1])
plt.title("Latent Space")
plt.xlabel("1")
plt.ylabel("2")
plt.show()


"""latents = results['latents']
normed  = normalize(latents, norm='l2')
labels = results['cluster_labels']
# UMAP to 2D
reducer   = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.5, random_state=42)
embedded  = reducer.fit_transform(normed)
"""

embedded = new_results['latents']
labels = new_results['cluster_labels']
plt.figure(figsize=(8, 6))
plt.scatter(embedded[:, 0], embedded[:, 1],
            c=labels, cmap='tab10', s=15, alpha=0.8)
plt.title("Clusters: quantile=0.1078, n_clusters={4}")
plt.colorbar(label='Cluster')
plt.xlabel("1")
plt.ylabel("2")
plt.show()

### Verify performance

In [ ]:
ground_truth_labels = np.load("rat_movement_large_labels.npy", allow_pickle = True)
raw_sequence = np.load("rat_movement_large.npy")
from scipy.stats import mode

In [ ]:
gt_window_labels = []
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment

# ── Parameters (must match pipeline) ──────────────────────────────────────
min_segment_frames = 30
max_segment_frames = 600

# ── Reconstruct window frame ranges ───────────────────────────────────────
n_frames   = len(raw_sequence)
boundaries = np.unique(
    np.concatenate([[0], results['transition_frames'], [n_frames]])
).astype(int)

gt_window_labels = []
for seg_idx in range(len(boundaries) - 1):
    seg_start = boundaries[seg_idx]
    seg_end   = boundaries[seg_idx + 1]
    seg_len   = seg_end - seg_start

    if seg_len < min_segment_frames:
        continue

    for chunk_start in range(0, seg_len, max_segment_frames):
        abs_start = seg_start + chunk_start
        abs_end   = min(abs_start + max_segment_frames, seg_end)
        chunk_len = abs_end - abs_start

        if chunk_len < min_segment_frames:
            continue

        window_frames = ground_truth_labels[abs_start:abs_end]
        values, counts = np.unique(window_frames, return_counts=True)
        majority = values[np.argmax(counts)]
        gt_window_labels.append(majority)

gt_window_labels = np.array(gt_window_labels)
cluster_labels   = results['cluster_labels']

print(f"GT labels:      {len(gt_window_labels)}")
print(f"Cluster labels: {len(cluster_labels)}")
assert len(gt_window_labels) == len(cluster_labels), \
    f"Mismatched: {len(gt_window_labels)} vs {len(cluster_labels)}"

# ── Convert GT strings to numeric for sklearn metrics ─────────────────────
behaviour_names = np.unique(gt_window_labels)   # e.g. ['explore' 'groom' ...]
beh_to_idx      = {b: i for i, b in enumerate(behaviour_names)}
gt_numeric      = np.array([beh_to_idx[b] for b in gt_window_labels])

# ── Metrics ────────────────────────────────────────────────────────────────
ari = adjusted_rand_score(gt_numeric, cluster_labels)
nmi = normalized_mutual_info_score(gt_numeric, cluster_labels)
print(f"\nARI:    {ari:.3f}  (1.0 = perfect, 0 = random)")
print(f"NMI:    {nmi:.3f}  (1.0 = perfect, 0 = random)")

# ── Build confusion matrix manually (GT rows, cluster columns) ────────────
cluster_ids = np.unique(cluster_labels)
cm = np.zeros((len(behaviour_names), len(cluster_ids)), dtype=int)

for gt, pred in zip(gt_window_labels, cluster_labels):
    cm[beh_to_idx[gt], pred] += 1

# ── Hungarian matching: optimal cluster → behaviour assignment ─────────────
row_ind, col_ind = linear_sum_assignment(-cm)
print("\nOptimal cluster → behaviour mapping:")
for r, c in zip(row_ind, col_ind):
    total    = cm[:, c].sum()
    correct  = cm[r, c]
    print(f"  Cluster {c:2d}  →  {behaviour_names[r]:14s}  "
          f"({correct}/{total} = {correct/total:.1%})")

# ── Purity ─────────────────────────────────────────────────────────────────
purity = np.sum(np.max(cm, axis=0)) / np.sum(cm)
print(f"\nPurity: {purity:.3f}")

# ── Pure windows ───────────────────────────────────────────────────────────
pure_windows = []
for seg_idx in range(len(boundaries) - 1):
    seg_start = boundaries[seg_idx]
    seg_end   = boundaries[seg_idx + 1]
    seg_len   = seg_end - seg_start

    if seg_len < min_segment_frames:
        continue

    for chunk_start in range(0, seg_len, max_segment_frames):
        abs_start = seg_start + chunk_start
        abs_end   = min(abs_start + max_segment_frames, seg_end)
        if (abs_end - abs_start) < min_segment_frames:
            continue
        n_unique = len(np.unique(ground_truth_labels[abs_start:abs_end]))
        pure_windows.append(n_unique == 1)

print(f"Pure windows:   {np.mean(pure_windows):.1%}")

# ── Plot confusion matrix ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
            xticklabels=[f'Cluster {i}' for i in cluster_ids],
            yticklabels=behaviour_names)
ax.set_xlabel("Predicted Cluster")
ax.set_ylabel("Ground Truth Behaviour")
ax.set_title(f"Confusion Matrix  (ARI={ari:.3f},  NMI={nmi:.3f},  Purity={purity:.3f})")
plt.tight_layout()
plt.show()

In [ ]:
# ── Check 1: how pure are your windows? ───────────────────
# if this is low, transition detection is the bottleneck
print(f"Pure windows: {np.mean(pure_windows):.1%}")
# if < 70%, fix transition detection before anything else

# ── Check 2: does the latent space have structure? ─────────
# plot GT labels on the latent space
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
scatter = axes[0].scatter(results['latents'][:, 0], results['latents'][:, 1],
                           c=gt_numeric, cmap='tab10', alpha=0.7, s=20)
axes[0].set_title("Ground Truth Behaviours")
plt.colorbar(scatter, ax=axes[0],
             ticks=range(len(behaviour_names))).set_ticklabels(behaviour_names)

scatter2 = axes[1].scatter(results['latents'][:, 0], results['latents'][:, 1],
                            c=results['cluster_labels'], cmap='tab10', alpha=0.7, s=20)
axes[1].set_title("Predicted Clusters")
plt.colorbar(scatter2, ax=axes[1])
plt.tight_layout()
plt.show()
# if GT colours are jumbled → model isn't learning, fix preprocessing/training
# if GT colours show structure but clusters don't align → fix clustering

# ── Check 3: upper bound ARI if windows were perfect ───────
# assign each window its majority GT label as the prediction
# this tells you the maximum ARI your transition detector allows
from sklearn.metrics import adjusted_rand_score
upper_bound_ari = adjusted_rand_score(gt_numeric, gt_numeric)
print(f"Upper bound ARI (perfect clustering): {upper_bound_ari:.3f}")  # should be 1.0

# more useful — what ARI would you get if you clustered perfectly
# on only the pure windows?
pure_mask = np.array(pure_windows)
if pure_mask.sum() > 0:
    ari_pure = adjusted_rand_score(gt_numeric[pure_mask],
                                   results['cluster_labels'][pure_mask])
    print(f"ARI on pure windows only: {ari_pure:.3f}")

In [ ]:
# ============================================================
# SAVE ALL RESULTS AND FIGURES TO ./results
# ============================================================


RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── 0. Save hyperparameter search log ─────────────────────────────────────
if 'search_log' in dir():
    with open(os.path.join(RESULTS_DIR, "hyperparameter_search.json"), "w") as f:
        json.dump({
            "objective": "maximize ARI",
            "max_iterations": 10,
            "percentile_range": [50, 100],
            "quantile_range": [0.05, 0.5],
            "search_log": search_log,
            "best_percentile": results.get('_percentile', None),
            "best_quantile": results.get('_quantile', None),
        }, f, indent=2)
    print("Saved hyperparameter search log.")

# ── 1. Save model weights ─────────────────────────────────────────────────
torch.save(results['model'].state_dict(), os.path.join(RESULTS_DIR, "model_stage1.pth"))
torch.save(results['model'].state_dict(), os.path.join(RESULTS_DIR, "model_stage2.pth"))
print("Saved model weights.")

# ── 2. Save numerical results ─────────────────────────────────────────────
np.save(os.path.join(RESULTS_DIR, "latents.npy"), results['latents'])
np.save(os.path.join(RESULTS_DIR, "latents_2d.npy"), results['latents_2d'])
np.save(os.path.join(RESULTS_DIR, "cluster_labels.npy"), results['cluster_labels'])
np.save(os.path.join(RESULTS_DIR, "transition_frames.npy"), results['transition_frames'])
np.save(os.path.join(RESULTS_DIR, "losses.npy"), results['losses'])
np.save(os.path.join(RESULTS_DIR, "positions.npy"), results['positions'])
np.save(os.path.join(RESULTS_DIR, "smoothed_losses.npy"), results['smoothed_losses'])
#np.save(os.path.join(RESULTS_DIR, "lossgraph_stage1.npy"), np.array(results['lossgraph_stage1']))
#np.save(os.path.join(RESULTS_DIR, "lossgraph_stage2.npy"), np.array(results['lossgraph_stage2']))
print("Saved numerical results.")

# ── 3. Save window metadata ───────────────────────────────────────────────
window_lengths = [len(w) for w in results['windows']]
window_meta = {
    "n_windows": len(results['windows']),
    "min_length": int(min(window_lengths)),
    "max_length": int(max(window_lengths)),
    "mean_length": float(np.mean(window_lengths)),
    "n_transitions": int(len(results['transition_frames'])),
    "n_clusters": int(len(np.unique(results['cluster_labels']))),
    "best_percentile": results.get('_percentile', None),
    "best_quantile": results.get('_quantile', None),
}
with open(os.path.join(RESULTS_DIR, "window_meta.json"), "w") as f:
    json.dump(window_meta, f, indent=2)
print("Saved window metadata.")

# ── 4. Save clustering metrics ────────────────────────────────────────────
behaviour_names = np.unique(gt_window_labels)
beh_to_idx = {b: i for i, b in enumerate(behaviour_names)}
gt_numeric = np.array([beh_to_idx[b] for b in gt_window_labels])
cluster_ids = np.unique(results['cluster_labels'])
cm = np.zeros((len(behaviour_names), len(cluster_ids)), dtype=int)
for gt, pred in zip(gt_window_labels, results['cluster_labels']):
    cm[beh_to_idx[gt], pred] += 1

ari = adjusted_rand_score(gt_numeric, results['cluster_labels'])
nmi = normalized_mutual_info_score(gt_numeric, results['cluster_labels'])
purity = np.sum(np.max(cm, axis=0)) / np.sum(cm)

row_ind, col_ind = linear_sum_assignment(-cm)
mapping = {}
for r, c in zip(row_ind, col_ind):
    total = int(cm[:, c].sum())
    correct = int(cm[r, c])
    mapping[f"Cluster {c}"] = {
        "behaviour": str(behaviour_names[r]),
        "correct": correct,
        "total": total,
        "accuracy": correct / total if total > 0 else 0.0,
    }

metrics = {
    "ARI": round(float(ari), 4),
    "NMI": round(float(nmi), 4),
    "Purity": round(float(purity), 4),
    "cluster_behaviour_mapping": mapping,
    "confusion_matrix": cm.tolist(),
    "behaviour_names": behaviour_names.tolist(),
    "cluster_ids": cluster_ids.tolist(),
}
with open(os.path.join(RESULTS_DIR, "clustering_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved clustering metrics.")

# ── 5. Save all figures ───────────────────────────────────────────────────

# 5a. Training loss curves
#fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#axes[0].plot(results['lossgraph_stage1'], 'b-', linewidth=1.5)
#axes[0].set_title("Stage 1: Fixed Windows Training Loss")
#axes[0].set_xlabel("Epoch")
#axes[0].set_ylabel("MSE Loss")
#axes[0].grid(True, alpha=0.3)

#axes[1].plot(results['lossgraph_stage2'], 'r-', linewidth=1.5)
#axes[1].set_title("Stage 2: Variable Windows Training Loss")
#axes[1].set_xlabel("Epoch")
#axes[1].set_ylabel("MSE Loss")
#axes[1].grid(True, alpha=0.3)

#plt.tight_layout()
#fig.savefig(os.path.join(RESULTS_DIR, "training_loss.png"), dpi=150, bbox_inches="tight")
#plt.close(fig)
#print("Saved training_loss.png")

# 5b. Reconstruction loss signal and transitions
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(results['positions'], results['losses'], 'b.', markersize=2, alpha=0.3, label="Raw loss")
ax.plot(results['positions'], results['smoothed_losses'], 'r-', linewidth=1.5, label="Smoothed loss")
for tf in results['transition_frames']:
    ax.axvline(x=tf, color='g', alpha=0.15, linewidth=0.5)
ax.set_title("Reconstruction Loss & Detected Transitions")
ax.set_xlabel("Frame")
ax.set_ylabel("MSE Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "reconstruction_loss.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved reconstruction_loss.png")

# 5c. Latent space scatter (no labels)
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(results['latents'][:, 0], results['latents'][:, 1], s=10, alpha=0.6)
ax.set_title("Latent Space")
ax.set_xlabel("Dim 1")
ax.set_ylabel("Dim 2")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "latent_space.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved latent_space.png")

# 5d. Clustered latent space
n_clusters = len(np.unique(results['cluster_labels']))
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(results['latents'][:, 0], results['latents'][:, 1],
                     c=results['cluster_labels'], cmap='tab10', s=15, alpha=0.8)
ax.set_title(f"Clusters (n={n_clusters})\n"
             f"percentile={results.get('_percentile', 'N/A'):.1f}, quantile={results.get('_quantile', 'N/A'):.3f}")
plt.colorbar(scatter, ax=ax, label='Cluster')
ax.set_xlabel("Dim 1")
ax.set_ylabel("Dim 2")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "clustered_latent_space.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved clustered_latent_space.png")

# 5e. Confusion matrix
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
            xticklabels=[f'Cluster {i}' for i in cluster_ids],
            yticklabels=behaviour_names)
ax.set_xlabel("Predicted Cluster")
ax.set_ylabel("Ground Truth Behaviour")
ax.set_title(f"Confusion Matrix (ARI={ari:.3f}, NMI={nmi:.3f}, Purity={purity:.3f})\n"
             f"percentile={results.get('_percentile', 'N/A'):.1f}, quantile={results.get('_quantile', 'N/A'):.3f}")
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved confusion_matrix.png")

# 5f. Diagnostic: GT vs Predicted on latent space
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
scatter = axes[0].scatter(results['latents'][:, 0], results['latents'][:, 1],
                          c=gt_numeric, cmap='tab10', alpha=0.7, s=20)
axes[0].set_title("Ground Truth Behaviours")
plt.colorbar(scatter, ax=axes[0],
             ticks=range(len(behaviour_names))).set_ticklabels(behaviour_names)

scatter2 = axes[1].scatter(results['latents'][:, 0], results['latents'][:, 1],
                           c=results['cluster_labels'], cmap='tab10', alpha=0.7, s=20)
axes[1].set_title("Predicted Clusters")
plt.colorbar(scatter2, ax=axes[1])
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "diagnostic_gt_vs_predicted.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved diagnostic_gt_vs_predicted.png")

# 5g. Pure windows diagnostic bar chart
pure_mask = np.array(pure_windows)
ari_pure = adjusted_rand_score(gt_numeric[pure_mask], results['cluster_labels'][pure_mask]) if pure_mask.sum() > 0 else 0
fig, ax = plt.subplots(figsize=(8, 5))
categories = ["All Windows", "Pure Windows Only"]
ari_values = [ari, ari_pure]
nmi_values = [nmi, nmi]
x = np.arange(len(categories))
width = 0.35
bars1 = ax.bar(x - width/2, ari_values, width, label='ARI', color='steelblue')
bars2 = ax.bar(x + width/2, nmi_values, width, label='NMI', color='coral')
ax.set_ylabel("Score")
ax.set_title("Clustering Performance: All vs Pure Windows")
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.set_ylim(0, 1.1)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "performance_comparison.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved performance_comparison.png")

print(f"\nAll results saved to ./{RESULTS_DIR}/")